### Configurations and Setups

In [1]:
# ============================================================
# Cell 1 — Configuration and fresh continuation folders
# ============================================================

from __future__ import annotations

import ast
import gc
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, get_dataset_config_names, get_dataset_split_names, load_dataset
from IPython.display import display
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_float32_matmul_precision("high")

PROJECT_DIR = Path(os.environ.get("AXMT_HOME", str(Path.home() / "alexandriax_mt_14d"))).expanduser()
BASE_MODEL_DIR = PROJECT_DIR / "models" / "hf" / "NileChat-3B-Base"

PARENT_EXPERIMENT_NAME = (
    "nilechat3b_alexandria_all14_context3_complete2shot_"
    "all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1"
)

PARENT_RUN_DIR = PROJECT_DIR / "runs" / "nilechat3b_all14" / PARENT_EXPERIMENT_NAME
PARENT_CHECKPOINT_STEP = 16600
PARENT_CHECKPOINT_DIR = PARENT_RUN_DIR / f"checkpoint-{PARENT_CHECKPOINT_STEP}"

NEW_EXPERIMENT_NAME = (
    "nilechat3b_dev12250_hftest60_from16600_"
    "complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1"
)

OUTPUT_DIR = PROJECT_DIR / "runs" / "nilechat3b_dev_continuation" / NEW_EXPERIMENT_NAME
DATA_RUN_DIR = PROJECT_DIR / "data" / "nilechat3b_dev_continuation" / NEW_EXPERIMENT_NAME
FINAL_ADAPTER_DIR = PROJECT_DIR / "models" / "final_adapters" / "nilechat3b_dev_continuation" / NEW_EXPERIMENT_NAME

for path in [OUTPUT_DIR, DATA_RUN_DIR, FINAL_ADAPTER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "UBC-NLP/alexandria"

PUBLIC_TEST_TRAIN_FRACTION = 0.60
PUBLIC_TEST_SELECTION_FRACTION = 0.40

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True
USE_FEW_SHOTS = True
N_FEW_SHOTS = 2
MAX_FEW_SHOT_EXAMPLE_CHARS = 450
MAX_SEQ_LENGTH = 2048

NUM_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

LOGGING_STEPS = 10
SAVE_STEPS = 100
EVAL_STEPS = 100
SAVE_TOTAL_LIMIT = 100

EXPECTED_ORIGINAL_TRAIN_TURNS = 66480
EXPECTED_PUBLIC_TEST_TURNS = 14442
EXPECTED_OFFICIAL_DEV_TURNS = 12250

assert torch.cuda.is_available(), "CUDA GPU is required."
assert BASE_MODEL_DIR.exists(), BASE_MODEL_DIR
assert PARENT_CHECKPOINT_DIR.exists(), PARENT_CHECKPOINT_DIR
assert (PARENT_CHECKPOINT_DIR / "adapter_config.json").exists()

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Base model:", BASE_MODEL_DIR)
print("Parent checkpoint:", PARENT_CHECKPOINT_DIR)
print("Fresh output folder:", OUTPUT_DIR)
print("Fresh data folder:", DATA_RUN_DIR)
print("Training split:", PUBLIC_TEST_TRAIN_FRACTION)
print("Selection split:", PUBLIC_TEST_SELECTION_FRACTION)
print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Save/eval every:", SAVE_STEPS)
print("Save limit:", SAVE_TOTAL_LIMIT)

GPU: NVIDIA GeForce RTX 5090
BF16 supported: True
Base model: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Parent checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Fresh output folder: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1
Fresh data folder: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1
Training split: 0.6
Selection split: 0.4
Epochs: 3
Learning rate: 2e-05
Save/eval every: 100
Save limit: 100


### **Data Loading and Preparation**

In [2]:
# ============================================================
# Cell 2 — Load and flatten train, labeled test, and DEV
# ============================================================

FLAT_CACHE_DIR = DATA_RUN_DIR / "flat_data_cache_v1"
FLAT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_TRAIN_CACHE = FLAT_CACHE_DIR / "original_train_66480.pkl"
PUBLIC_TEST_CACHE = FLAT_CACHE_DIR / "public_labeled_test_14442.pkl"
OFFICIAL_DEV_CACHE = FLAT_CACHE_DIR / "official_dev_12250.pkl"

def to_plain(value):
    if isinstance(value, np.ndarray):
        return [to_plain(item) for item in value.tolist()]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, tuple):
        return [to_plain(item) for item in value]
    if isinstance(value, list):
        return [to_plain(item) for item in value]
    if isinstance(value, dict):
        return {str(key): to_plain(item) for key, item in value.items()}
    return value

def scalar_text(value):
    value = to_plain(value)

    if value is None:
        return ""

    if isinstance(value, (list, dict)):
        return str(value).strip()

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()

def normalize_turn_list(value):
    value = to_plain(value)

    if value is None:
        return []

    if isinstance(value, list):
        normalized = []

        for item in value:
            item = to_plain(item)

            if isinstance(item, dict):
                normalized.append(item)
            elif item is not None:
                normalized.append({"text": scalar_text(item)})

        return normalized

    if isinstance(value, dict):
        lengths = [len(item) for item in value.values() if isinstance(to_plain(item), list)]

        if not lengths:
            return []

        rows = []

        for row_index in range(max(lengths)):
            item = {}

            for key, values in value.items():
                values = to_plain(values)

                if isinstance(values, list):
                    item[key] = values[row_index] if row_index < len(values) else None
                else:
                    item[key] = values

            rows.append(item)

        return rows

    return []

def turn_field(turn, possible_keys, default=""):
    turn = to_plain(turn)

    if not isinstance(turn, dict):
        return default

    for key in possible_keys:
        if key not in turn:
            continue

        value = scalar_text(turn[key])

        if value:
            return value

    return default

def turn_text(turn):
    return turn_field(
        turn,
        ["text", "source", "source_text", "english", "english_text", "sentence", "utterance", "content", "value", "translation"],
        default="",
    )

def extract_turn_order(turn, fallback_index):
    raw_value = turn_field(turn, ["turn_order", "turn_id", "order", "idx", "index"], default="")

    if raw_value:
        try:
            return int(raw_value)
        except Exception:
            pass

    return int(fallback_index + 1)

def flatten_split(dataset, config_name, split_name):
    records = []

    for conversation_index, raw_row in enumerate(dataset):
        row = to_plain(dict(raw_row))

        conversation_id = scalar_text(
            row.get("conv_id", row.get("conversation_id", f"{config_name}_{split_name}_{conversation_index}"))
        )

        country = scalar_text(row.get("country", config_name)) or config_name
        dialect = scalar_text(row.get("dialect", ""))
        domain = scalar_text(row.get("domain", ""))
        participants = scalar_text(row.get("participants", ""))
        persona = scalar_text(row.get("persona", row.get("roles", row.get("speaker_roles", ""))))
        translator_id = scalar_text(row.get("translator_id", ""))
        reviewer_id = scalar_text(row.get("reviewer_id", ""))

        english_turns = normalize_turn_list(row.get("english_conversation", []))
        arabic_turns = normalize_turn_list(row.get("dialectal_conversation", []))

        if len(english_turns) != len(arabic_turns):
            raise RuntimeError(
                f"Turn-count mismatch in {config_name}/{split_name}/{conversation_id}: "
                f"{len(english_turns)} English versus {len(arabic_turns)} Arabic"
            )

        for turn_index, (english_turn, arabic_turn) in enumerate(zip(english_turns, arabic_turns)):
            source_text = turn_text(english_turn)
            target_arabic = turn_text(arabic_turn)

            if not source_text or not target_arabic:
                continue

            turn_order = extract_turn_order(english_turn, turn_index)
            arabic_turn_order = extract_turn_order(arabic_turn, turn_index)

            if turn_order != arabic_turn_order:
                raise RuntimeError(
                    f"Turn-order mismatch in {config_name}/{split_name}/{conversation_id}: "
                    f"{turn_order} versus {arabic_turn_order}"
                )

            previous_english_turns = []
            previous_start = max(0, turn_index - MAX_CONTEXT_TURNS)

            for previous_index in range(previous_start, turn_index):
                previous_turn = english_turns[previous_index]

                previous_english_turns.append(
                    {
                        "turn_order": extract_turn_order(previous_turn, previous_index),
                        "speaker": turn_field(previous_turn, ["speaker", "role", "speaker_role", "participant"], ""),
                        "direction": turn_field(
                            previous_turn,
                            ["direction", "gender_direction", "speaker_addressee_gender"],
                            "",
                        ),
                        "text": turn_text(previous_turn),
                    }
                )

            records.append(
                {
                    "source_id": f"{config_name}_{split_name}_{conversation_id}_{turn_order}",
                    "config": config_name,
                    "country": country,
                    "split": split_name,
                    "conversation_id": conversation_id,
                    "turn_order": int(turn_order),
                    "turn_id": int(turn_order),
                    "dialect": dialect,
                    "domain": domain,
                    "participants": participants,
                    "persona": persona,
                    "speaker": turn_field(english_turn, ["speaker", "role", "speaker_role", "participant"], ""),
                    "gender_direction": turn_field(
                        english_turn,
                        ["direction", "gender_direction", "speaker_addressee_gender"],
                        "",
                    ),
                    "previous_english_turns": previous_english_turns,
                    "source_text": source_text,
                    "target_arabic": target_arabic,
                    "translator_id": translator_id,
                    "reviewer_id": reviewer_id,
                }
            )

    return records

if ORIGINAL_TRAIN_CACHE.exists() and PUBLIC_TEST_CACHE.exists() and OFFICIAL_DEV_CACHE.exists():
    print("Loading flattened data from the fresh cache.")
    original_train_df = pd.read_pickle(ORIGINAL_TRAIN_CACHE)
    public_test_df = pd.read_pickle(PUBLIC_TEST_CACHE)
    official_dev_df = pd.read_pickle(OFFICIAL_DEV_CACHE)
else:
    available_configs = sorted(get_dataset_config_names(DATASET_NAME))
    collected = {"train": [], "test": [], "dev": []}

    for config_name in available_configs:
        split_names = set(get_dataset_split_names(DATASET_NAME, config_name))

        for split_name in ["train", "test", "dev"]:
            if split_name not in split_names:
                print(f"Skipping {config_name}/{split_name}: unavailable")
                continue

            print(f"Loading {config_name}/{split_name}")
            dataset = load_dataset(DATASET_NAME, name=config_name, split=split_name)
            records = flatten_split(dataset, config_name, split_name)
            collected[split_name].extend(records)
            print("Turns:", len(records))

    original_train_df = pd.DataFrame(collected["train"])
    public_test_df = pd.DataFrame(collected["test"])
    official_dev_df = pd.DataFrame(collected["dev"])

    original_train_df.to_pickle(ORIGINAL_TRAIN_CACHE)
    public_test_df.to_pickle(PUBLIC_TEST_CACHE)
    official_dev_df.to_pickle(OFFICIAL_DEV_CACHE)

for frame in [original_train_df, public_test_df, official_dev_df]:
    frame["source_id"] = frame["source_id"].astype(str)
    frame["config"] = frame["config"].astype(str)
    frame["conversation_id"] = frame["conversation_id"].astype(str)
    frame["source_text"] = frame["source_text"].fillna("").astype(str).str.strip()
    frame["target_arabic"] = frame["target_arabic"].fillna("").astype(str).str.strip()

assert len(original_train_df) == EXPECTED_ORIGINAL_TRAIN_TURNS, len(original_train_df)
assert len(public_test_df) == EXPECTED_PUBLIC_TEST_TURNS, len(public_test_df)
assert len(official_dev_df) == EXPECTED_OFFICIAL_DEV_TURNS, len(official_dev_df)

for name, frame in [
    ("original train", original_train_df),
    ("public labeled test", public_test_df),
    ("official DEV", official_dev_df),
]:
    assert not frame["source_id"].duplicated().any(), f"Duplicate source_id in {name}"
    assert not frame["source_text"].eq("").any(), f"Empty source in {name}"
    assert not frame["target_arabic"].eq("").any(), f"Empty target in {name}"

print("\nOriginal train:", original_train_df.shape)
print("Public labeled test:", public_test_df.shape)
print("Official DEV:", official_dev_df.shape)

print("\nOriginal train countries:", sorted(original_train_df["config"].unique()))
print("Public labeled test countries:", sorted(public_test_df["config"].unique()))
print("Official DEV countries:", sorted(official_dev_df["config"].unique()))

Loading flattened data from the fresh cache.

Original train: (66480, 18)
Public labeled test: (14442, 18)
Official DEV: (12250, 18)

Original train countries: ['EG', 'JO', 'LB', 'MA', 'MR', 'OM', 'PS', 'SA', 'SY', 'TN', 'YE']
Public labeled test countries: ['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']
Official DEV countries: ['EG', 'JO', 'LB', 'MA', 'MR', 'OM', 'PS', 'SA', 'SY', 'TN', 'YE']


In [3]:
# ============================================================
# Cell 3 — Random 60/40 split by complete conversations
# ============================================================

SPLIT_ASSIGNMENT_PATH = DATA_RUN_DIR / "public_test_grouped_60_40_assignments.csv"
FINE_TUNE_ROWS_PATH = DATA_RUN_DIR / "fine_tune_rows_dev_plus_public60.pkl"
SELECTION_ROWS_PATH = DATA_RUN_DIR / "selection_rows_public40.pkl"

def stable_country_seed(country, base_seed=SEED):
    digest = hashlib.md5(f"{country}_{base_seed}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16)

def grouped_country_split(frame, train_fraction):
    conversation_table = (
        frame.groupby(["config", "conversation_id"], as_index=False)
        .agg(
            n_turns=("source_id", "size"),
            domain=("domain", "first"),
            dialect=("dialect", "first"),
        )
    )

    train_keys = set()

    for country, country_conversations in conversation_table.groupby("config", sort=True):
        if len(country_conversations) < 2:
            raise RuntimeError(f"Not enough conversations to split country {country}")

        shuffled = country_conversations.sample(
            frac=1.0,
            random_state=stable_country_seed(country),
        ).reset_index(drop=True)

        target_train_turns = shuffled["n_turns"].sum() * train_fraction
        cumulative_turns = shuffled["n_turns"].cumsum().to_numpy()

        possible_counts = np.arange(1, len(shuffled))
        best_count = int(
            possible_counts[
                np.argmin(np.abs(cumulative_turns[:-1] - target_train_turns))
            ]
        )

        selected = shuffled.iloc[:best_count]

        for row in selected.itertuples(index=False):
            train_keys.add((str(row.config), str(row.conversation_id)))

    row_keys = list(zip(frame["config"].astype(str), frame["conversation_id"].astype(str)))
    train_mask = pd.Series([key in train_keys for key in row_keys], index=frame.index)

    return frame.loc[train_mask].copy(), frame.loc[~train_mask].copy()

if SPLIT_ASSIGNMENT_PATH.exists() and FINE_TUNE_ROWS_PATH.exists() and SELECTION_ROWS_PATH.exists():
    print("Loading the locked 60/40 split.")
    assignment_df = pd.read_csv(SPLIT_ASSIGNMENT_PATH)
    fine_tune_df = pd.read_pickle(FINE_TUNE_ROWS_PATH)
    selection_df = pd.read_pickle(SELECTION_ROWS_PATH)

    public_train_ids = set(
        assignment_df.loc[assignment_df["partition"] == "public_train_60", "source_id"].astype(str)
    )
    public_train_df = public_test_df[public_test_df["source_id"].isin(public_train_ids)].copy()
else:
    public_train_df, selection_df = grouped_country_split(
        public_test_df,
        PUBLIC_TEST_TRAIN_FRACTION,
    )

    fine_tune_df = pd.concat(
        [official_dev_df, public_train_df],
        ignore_index=True,
    )

    assignment_df = public_test_df[
        ["source_id", "config", "conversation_id", "turn_order"]
    ].copy()

    public_train_ids = set(public_train_df["source_id"].astype(str))
    assignment_df["partition"] = np.where(
        assignment_df["source_id"].astype(str).isin(public_train_ids),
        "public_train_60",
        "selection_40",
    )

    assignment_df.to_csv(SPLIT_ASSIGNMENT_PATH, index=False, encoding="utf-8-sig")
    fine_tune_df.to_pickle(FINE_TUNE_ROWS_PATH)
    selection_df.to_pickle(SELECTION_ROWS_PATH)

fine_tune_df = fine_tune_df.reset_index(drop=True)
selection_df = selection_df.reset_index(drop=True)
public_train_df = public_train_df.reset_index(drop=True)

public_train_conversations = set(
    zip(public_train_df["config"], public_train_df["conversation_id"])
)
selection_conversations = set(
    zip(selection_df["config"], selection_df["conversation_id"])
)

assert not public_train_conversations.intersection(selection_conversations)
assert not set(public_train_df["source_id"]).intersection(selection_df["source_id"])
assert len(public_train_df) + len(selection_df) == EXPECTED_PUBLIC_TEST_TURNS
assert len(fine_tune_df) == len(official_dev_df) + len(public_train_df)
assert not fine_tune_df["source_id"].duplicated().any()
assert sorted(selection_df["config"].unique()) == sorted(public_test_df["config"].unique())

dev_conversations = set(zip(official_dev_df["config"], official_dev_df["conversation_id"]))
unexpected_dev_overlap = dev_conversations.intersection(selection_conversations)

if unexpected_dev_overlap:
    raise RuntimeError(
        f"DEV and selection contain overlapping conversation IDs: "
        f"{list(unexpected_dev_overlap)[:10]}"
    )

split_summary = pd.concat(
    [
        public_test_df.groupby("config").size().rename("total_turns"),
        public_train_df.groupby("config").size().rename("train_turns"),
        selection_df.groupby("config").size().rename("selection_turns"),
    ],
    axis=1,
).fillna(0).astype(int).reset_index()

split_summary["actual_train_fraction"] = (
    split_summary["train_turns"] / split_summary["total_turns"]
)

print("Fine-tuning rows:", len(fine_tune_df))
print("  Official DEV:", len(official_dev_df))
print("  Public 60%:", len(public_train_df))
print("Selection rows:", len(selection_df))
print("Fine-tuning countries:", sorted(fine_tune_df["config"].unique()))
print("Selection countries:", sorted(selection_df["config"].unique()))

display(split_summary)

Loading the locked 60/40 split.
Fine-tuning rows: 20920
  Official DEV: 12250
  Public 60%: 8670
Selection rows: 5772
Fine-tuning countries: ['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']
Selection countries: ['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']


,config,total_turns,train_turns,selection_turns,actual_train_fraction
0,EG,1118,671,447,0.600179
1,JO,1107,666,441,0.601626
2,LB,1106,664,442,0.600362
3,LY,1109,665,444,0.599639
4,MA,1115,668,447,0.599103
5,MR,1112,668,444,0.600719
6,OM,1118,670,448,0.599284
7,PS,1109,666,443,0.600541
8,SA,1113,668,445,0.600180
9,SD,1106,664,442,0.600362


### **Build the training-style two-shot prompts**

In [4]:
# ============================================================
# Cell 4 — Build deterministic complete two-shot prompts
# ============================================================

TRAIN_PROMPT_CACHE = DATA_RUN_DIR / "fine_tune_prompt_rows.pkl"
SELECTION_PROMPT_CACHE = DATA_RUN_DIR / "selection_prompt_rows.pkl"

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []

    for index, turn in enumerate(previous_turns, start=1):
        speaker = str(turn.get("speaker", "")).strip()
        text = str(turn.get("text", "")).strip()

        if speaker:
            lines.append(f"{index}. {speaker}: {text}")
        else:
            lines.append(f"{index}. {text}")

    return "\n".join(lines) if lines else "No previous context."

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []

    for name, value in fields:
        value = str(value).strip()

        if value:
            lines.append(f"{name}: {value}")

    return "\n".join(lines) if lines else "No metadata."

def build_few_shot_block(examples):
    if not USE_FEW_SHOTS or not examples:
        return "No examples available."

    blocks = []

    for index, example in enumerate(examples, start=1):
        metadata = []

        if str(example.get("config", "")).strip():
            metadata.append(f"config={example['config']}")

        if str(example.get("dialect", "")).strip():
            metadata.append(f"dialect={example['dialect']}")

        if str(example.get("domain", "")).strip():
            metadata.append(f"domain={example['domain']}")

        metadata_text = ", ".join(metadata) if metadata else "no metadata"

        blocks.append(
            f"""Example {index} ({metadata_text})
English:
{str(example["source_text"]).strip()}

Arabic:
{str(example["target_arabic"]).strip()}"""
        )

    return "\n\n".join(blocks)

def make_user_prompt(row):
    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{build_few_shot_block(row.get("few_shot_examples", []))}

Metadata:
{build_metadata_block(row)}

Previous English dialogue context:
{build_context(row.get("previous_english_turns", []))}

Current English turn:
{str(row["source_text"]).strip()}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": str(row["target_arabic"]).strip()},
    ]

def deterministic_row_seed(source_id):
    digest = hashlib.md5(f"{source_id}_{SEED}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16)

if TRAIN_PROMPT_CACHE.exists() and SELECTION_PROMPT_CACHE.exists():
    print("Loading prompt rows from cache.")
    train_prompt_df = pd.read_pickle(TRAIN_PROMPT_CACHE)
    selection_prompt_df = pd.read_pickle(SELECTION_PROMPT_CACHE)

    assert set(train_prompt_df["source_id"]) == set(fine_tune_df["source_id"])
    assert set(selection_prompt_df["source_id"]) == set(selection_df["source_id"])
else:
    pool_columns = [
        "source_id",
        "config",
        "dialect",
        "domain",
        "source_text",
        "target_arabic",
    ]

    few_shot_pool_df = pd.concat(
        [
            original_train_df[pool_columns],
            fine_tune_df[pool_columns],
        ],
        ignore_index=True,
    )

    for column in pool_columns:
        few_shot_pool_df[column] = few_shot_pool_df[column].fillna("").astype(str)

    few_shot_pool_df["fewshot_total_chars"] = (
        few_shot_pool_df["source_text"].str.len()
        + few_shot_pool_df["target_arabic"].str.len()
    )

    short_pool_df = few_shot_pool_df[
        few_shot_pool_df["fewshot_total_chars"] <= MAX_FEW_SHOT_EXAMPLE_CHARS
    ].copy()

    def build_record_groups(frame, group_columns):
        groups = {}

        for key, group in frame.groupby(group_columns, sort=False):
            groups[key] = group[pool_columns].to_dict("records")

        return groups

    short_by_config_domain = build_record_groups(short_pool_df, ["config", "domain"])
    short_by_config = build_record_groups(short_pool_df, "config")
    all_by_config_domain = build_record_groups(few_shot_pool_df, ["config", "domain"])
    all_by_config = build_record_groups(few_shot_pool_df, "config")

    def select_two_shots(row):
        if not USE_FEW_SHOTS:
            return []

        source_id = str(row["source_id"])
        config = str(row["config"])
        domain = str(row.get("domain", ""))

        candidate_lists = [
            short_by_config_domain.get((config, domain), []),
            short_by_config.get(config, []),
            all_by_config_domain.get((config, domain), []),
            all_by_config.get(config, []),
        ]

        for candidates in candidate_lists:
            valid_candidates = [
                candidate
                for candidate in candidates
                if str(candidate["source_id"]) != source_id
            ]

            if not valid_candidates:
                continue

            sample_size = min(N_FEW_SHOTS, len(valid_candidates))
            rng = random.Random(deterministic_row_seed(source_id))
            return rng.sample(valid_candidates, sample_size)

        raise RuntimeError(f"No few-shot candidates for {source_id}")

    train_prompt_df = fine_tune_df.copy()
    selection_prompt_df = selection_df.copy()

    print("Selecting training few-shots.")
    train_prompt_df["few_shot_examples"] = [
        select_two_shots(row)
        for row in tqdm(
            train_prompt_df.to_dict("records"),
            total=len(train_prompt_df),
            desc="Training prompts",
        )
    ]

    print("Selecting selection-set few-shots.")
    selection_prompt_df["few_shot_examples"] = [
        select_two_shots(row)
        for row in tqdm(
            selection_prompt_df.to_dict("records"),
            total=len(selection_prompt_df),
            desc="Selection prompts",
        )
    ]

    train_prompt_df["messages"] = [
        row_to_messages(row)
        for row in tqdm(
            train_prompt_df.to_dict("records"),
            total=len(train_prompt_df),
            desc="Training messages",
        )
    ]

    selection_prompt_df["messages"] = [
        row_to_messages(row)
        for row in tqdm(
            selection_prompt_df.to_dict("records"),
            total=len(selection_prompt_df),
            desc="Selection messages",
        )
    ]

    train_prompt_df.to_pickle(TRAIN_PROMPT_CACHE)
    selection_prompt_df.to_pickle(SELECTION_PROMPT_CACHE)

print("Training prompt rows:", len(train_prompt_df))
print("Selection prompt rows:", len(selection_prompt_df))
print("Example few-shot IDs:", [
    example["source_id"]
    for example in train_prompt_df.iloc[0]["few_shot_examples"]
])

Loading prompt rows from cache.
Training prompt rows: 20920
Selection prompt rows: 5772
Example few-shot IDs: ['EG_dev_B15-1-0-495_3', 'EG_train_B11-1-0-466_1']


### **Load Best LoRA Checkpoint-16600**

In [5]:
# ============================================================
# Cell 5 — Load base model and trainable continuation adapter
# ============================================================

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.trainer_utils import get_last_checkpoint

last_new_checkpoint = get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None

if last_new_checkpoint:
    ADAPTER_INITIALIZATION_PATH = Path(last_new_checkpoint)
    print("Existing continuation checkpoint found:", ADAPTER_INITIALIZATION_PATH)
else:
    ADAPTER_INITIALIZATION_PATH = PARENT_CHECKPOINT_DIR
    print("Starting new continuation from:", ADAPTER_INITIALIZATION_PATH)

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
    use_fast=True,
    extra_special_tokens={},
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_INITIALIZATION_PATH,
    is_trainable=True,
    local_files_only=True,
)

model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model.gradient_checkpointing_enable()

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

print("Loaded adapter:", ADAPTER_INITIALIZATION_PATH)
print("Active adapters:", list(model.peft_config))
print("dtype:", dtype)
print("4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("8-bit:", getattr(model, "is_loaded_in_8bit", False))
print("\nTrainable parameters:")
model.print_trainable_parameters()

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert trainable_parameters > 0, "The loaded adapter is not trainable."
assert len(model.peft_config) == 1, "More than one LoRA adapter is active."

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Existing continuation checkpoint found: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-7845


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded adapter: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-7845
Active adapters: ['default']
dtype: torch.bfloat16
4-bit: False
8-bit: False

Trainable parameters:
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


### **Training Preparation and SFT Formatting**

In [6]:
# ============================================================
# Cell 6 — Manual SFT format and token-length validation
# ============================================================

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

def get_message_content(messages, role):
    for message in messages:
        if message.get("role") == role:
            return message.get("content", "")
    return ""

def format_sft_text(messages):
    system_text = get_message_content(messages, "system").strip()
    user_text = get_message_content(messages, "user").strip()
    assistant_text = get_message_content(messages, "assistant").strip()

    text = (
        f"{SYSTEM_MARKER}\n{system_text}\n\n"
        f"{INSTRUCTION_MARKER}\n{user_text}\n\n"
        f"{RESPONSE_MARKER}\n{assistant_text}"
    )

    if tokenizer.eos_token is not None:
        text += tokenizer.eos_token

    return text

print("Formatting training text.")
train_texts = [
    format_sft_text(messages)
    for messages in tqdm(
        train_prompt_df["messages"],
        total=len(train_prompt_df),
        desc="Training text",
    )
]

print("Formatting selection text.")
selection_texts = [
    format_sft_text(messages)
    for messages in tqdm(
        selection_prompt_df["messages"],
        total=len(selection_prompt_df),
        desc="Selection text",
    )
]

assert all(RESPONSE_MARKER in text for text in train_texts)
assert all(RESPONSE_MARKER in text for text in selection_texts)

train_dataset_text = Dataset.from_dict({"text": train_texts})
eval_dataset_text = Dataset.from_dict({"text": selection_texts})

def count_tokens_batch(batch):
    encoded = tokenizer(
        batch["text"],
        add_special_tokens=False,
        truncation=False,
    )
    return {"n_tokens": [len(input_ids) for input_ids in encoded["input_ids"]]}

train_dataset_text = train_dataset_text.map(
    count_tokens_batch,
    batched=True,
    batch_size=64,
    desc="Counting training tokens",
)

eval_dataset_text = eval_dataset_text.map(
    count_tokens_batch,
    batched=True,
    batch_size=64,
    desc="Counting selection tokens",
)

def token_summary(dataset, name):
    lengths = pd.Series(dataset["n_tokens"])

    print(f"\n{name}:")
    print("Rows:", len(lengths))
    print("Median:", int(lengths.median()))
    print("P90:", int(lengths.quantile(0.90)))
    print("P95:", int(lengths.quantile(0.95)))
    print("P99:", int(lengths.quantile(0.99)))
    print("Maximum:", int(lengths.max()))
    print("Longer than MAX_SEQ_LENGTH:", int((lengths > MAX_SEQ_LENGTH).sum()))

token_summary(train_dataset_text, "Training token lengths")
token_summary(eval_dataset_text, "Selection token lengths")

too_long_train = sum(length > MAX_SEQ_LENGTH for length in train_dataset_text["n_tokens"])
too_long_eval = sum(length > MAX_SEQ_LENGTH for length in eval_dataset_text["n_tokens"])

if too_long_train or too_long_eval:
    print(
        "\nWARNING: Some rows exceed MAX_SEQ_LENGTH and will be truncated. "
        "Inspect the counts before training."
    )

train_dataset_text = train_dataset_text.remove_columns("n_tokens")
eval_dataset_text = eval_dataset_text.remove_columns("n_tokens")

print("\nTraining dataset:", train_dataset_text)
print("Selection dataset:", eval_dataset_text)
print("\nFormatted example:\n")
print(train_dataset_text[0]["text"][:3000])

Formatting training text.


Training text:   0%|          | 0/20920 [00:00<?, ?it/s]

Formatting selection text.


Selection text:   0%|          | 0/5772 [00:00<?, ?it/s]

Counting training tokens:   0%|          | 0/20920 [00:00<?, ? examples/s]

Counting selection tokens:   0%|          | 0/5772 [00:00<?, ? examples/s]


Training token lengths:
Rows: 20920
Median: 461
P90: 527
P95: 547
P99: 584
Maximum: 667
Longer than MAX_SEQ_LENGTH: 0

Selection token lengths:
Rows: 5772
Median: 461
P90: 528
P95: 547
P99: 578
Maximum: 672
Longer than MAX_SEQ_LENGTH: 0

Training dataset: Dataset({
    features: ['text'],
    num_rows: 20920
})
Selection dataset: Dataset({
    features: ['text'],
    num_rows: 5772
})

Formatted example:

### System:
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.

### Instruction:
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
Example 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)
English:
Not too deep, just enough for the water to flow freely without flooding the rows. Like

In [7]:
# ============================================================
# Cell 7 — Standard Transformers response-only trainer
# No Unsloth and no TRL dependency
# ============================================================

from transformers import Trainer, TrainingArguments

RESPONSE_TEMPLATE = f"{RESPONSE_MARKER}\n"

def tokenize_response_only(example):
    text = str(example["text"])

    if RESPONSE_TEMPLATE not in text:
        raise ValueError("Response marker is missing from a formatted example.")

    prompt_without_target, target_text = text.rsplit(RESPONSE_TEMPLATE, 1)
    prompt_text = prompt_without_target + RESPONSE_TEMPLATE

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    target_ids = tokenizer(
        target_text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    if not target_ids:
        raise ValueError("An example has an empty tokenized target.")

    was_truncated = len(prompt_ids) + len(target_ids) > MAX_SEQ_LENGTH

    if len(target_ids) >= MAX_SEQ_LENGTH:
        target_ids = target_ids[:MAX_SEQ_LENGTH]
        prompt_ids = []
    else:
        available_prompt_tokens = MAX_SEQ_LENGTH - len(target_ids)

        if len(prompt_ids) > available_prompt_tokens:
            prompt_ids = prompt_ids[-available_prompt_tokens:]

    input_ids = prompt_ids + target_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + target_ids.copy()

    supervised_tokens = sum(label != -100 for label in labels)

    if supervised_tokens == 0:
        raise ValueError("Response-only masking removed every target token.")

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "supervised_tokens": supervised_tokens,
        "was_truncated": was_truncated,
    }

print("Tokenizing and masking the training dataset.")

tokenized_train_dataset = train_dataset_text.map(
    tokenize_response_only,
    remove_columns=train_dataset_text.column_names,
    desc="Tokenizing training rows",
)

print("Tokenizing and masking the selection dataset.")

tokenized_eval_dataset = eval_dataset_text.map(
    tokenize_response_only,
    remove_columns=eval_dataset_text.column_names,
    desc="Tokenizing selection rows",
)

train_truncated = int(sum(tokenized_train_dataset["was_truncated"]))
eval_truncated = int(sum(tokenized_eval_dataset["was_truncated"]))

train_supervised = pd.Series(tokenized_train_dataset["supervised_tokens"])
eval_supervised = pd.Series(tokenized_eval_dataset["supervised_tokens"])

print("\nResponse-only tokenization summary:")
print("Training rows:", len(tokenized_train_dataset))
print("Selection rows:", len(tokenized_eval_dataset))
print("Truncated training rows:", train_truncated)
print("Truncated selection rows:", eval_truncated)
print("Training median supervised tokens:", int(train_supervised.median()))
print("Selection median supervised tokens:", int(eval_supervised.median()))
print("Minimum training supervised tokens:", int(train_supervised.min()))
print("Minimum selection supervised tokens:", int(eval_supervised.min()))

assert int(train_supervised.min()) > 0
assert int(eval_supervised.min()) > 0

keep_columns = {"input_ids", "attention_mask", "labels"}

tokenized_train_dataset = tokenized_train_dataset.remove_columns(
    [
        column
        for column in tokenized_train_dataset.column_names
        if column not in keep_columns
    ]
)

tokenized_eval_dataset = tokenized_eval_dataset.remove_columns(
    [
        column
        for column in tokenized_eval_dataset.column_names
        if column not in keep_columns
    ]
)

class ResponseOnlyDataCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        input_features = [
            {
                "input_ids": feature["input_ids"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ]

        batch = self.tokenizer.pad(
            input_features,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        padded_length = batch["input_ids"].shape[1]
        padded_labels = []

        for feature in features:
            labels = list(feature["labels"])
            padding_length = padded_length - len(labels)

            if self.tokenizer.padding_side == "right":
                labels = labels + [-100] * padding_length
            else:
                labels = [-100] * padding_length + labels

            padded_labels.append(labels)

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long,
        )

        return batch

data_collator = ResponseOnlyDataCollator(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=False,
    save_safetensors=True,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    eval_accumulation_steps=8,
    prediction_loss_only=True,
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_train_dataset,
    "eval_dataset": tokenized_eval_dataset,
    "data_collator": data_collator,
}

try:
    trainer = Trainer(
        **trainer_kwargs,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = Trainer(
        **trainer_kwargs,
        tokenizer=tokenizer,
    )

sample = tokenized_train_dataset[0]
sample_supervised_tokens = sum(
    label != -100
    for label in sample["labels"]
)

updates_per_epoch = math.ceil(
    len(tokenized_train_dataset)
    / (
        PER_DEVICE_TRAIN_BATCH_SIZE
        * GRAD_ACCUM_STEPS
    )
)

expected_total_updates = updates_per_epoch * NUM_EPOCHS
expected_periodic_checkpoints = expected_total_updates // SAVE_STEPS

print("\nTrainer ready without Unsloth.")
print("Training rows:", len(tokenized_train_dataset))
print("Selection rows:", len(tokenized_eval_dataset))
print("Supervised tokens in first example:", sample_supervised_tokens)
print("Updates per epoch:", updates_per_epoch)
print("Expected total updates:", expected_total_updates)
print("Expected periodic checkpoints:", expected_periodic_checkpoints)
print("Save/eval every:", SAVE_STEPS)
print("Save limit:", SAVE_TOTAL_LIMIT)
print("Parent checkpoint:", PARENT_CHECKPOINT_DIR)
print("New-run resume checkpoint:", last_new_checkpoint)

if expected_periodic_checkpoints > SAVE_TOTAL_LIMIT:
    print(
        "\nNOTE: save_total_limit will rotate out approximately "
        f"{expected_periodic_checkpoints - SAVE_TOTAL_LIMIT} "
        "early continuation checkpoints."
    )

print(
    "\nThe labels before the Arabic response are masked with -100. "
    "Only the target Arabic translation contributes to the loss."
)

Tokenizing and masking the training dataset.


Tokenizing training rows:   0%|          | 0/20920 [00:00<?, ? examples/s]

Tokenizing and masking the selection dataset.


Tokenizing selection rows:   0%|          | 0/5772 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.



Response-only tokenization summary:
Training rows: 20920
Selection rows: 5772
Truncated training rows: 0
Truncated selection rows: 0
Training median supervised tokens: 31
Selection median supervised tokens: 31
Minimum training supervised tokens: 2
Minimum selection supervised tokens: 3

Trainer ready without Unsloth.
Training rows: 20920
Selection rows: 5772
Supervised tokens in first example: 27
Updates per epoch: 2615
Expected total updates: 7845
Expected periodic checkpoints: 78
Save/eval every: 100
Save limit: 100
Parent checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
New-run resume checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-7845

The labels before the Arabic response are masked with -100. Only

### **Training**

In [8]:
# ============================================================
# Cell 8 — Train or resume inside the fresh continuation folder
# ============================================================

TRAINING_SUMMARY_PATH = DATA_RUN_DIR / "training_summary.json"

if last_new_checkpoint:
    print("Resuming the new continuation run from:", last_new_checkpoint)
    trainer_stats = trainer.train(resume_from_checkpoint=last_new_checkpoint)
else:
    print("Starting the new continuation run from checkpoint 16600 weights.")
    trainer_stats = trainer.train()

trainer.save_state()
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

training_summary = {
    "new_experiment_name": NEW_EXPERIMENT_NAME,
    "parent_checkpoint": str(PARENT_CHECKPOINT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    "official_dev_rows": int(len(official_dev_df)),
    "public_train_rows": int(len(public_train_df)),
    "fine_tune_rows": int(len(fine_tune_df)),
    "selection_rows": int(len(selection_df)),
    "public_train_fraction": PUBLIC_TEST_TRAIN_FRACTION,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "trainer_metrics": trainer_stats.metrics,
    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open(TRAINING_SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(training_summary, file, ensure_ascii=False, indent=2, default=str)

print("\nTraining completed.")
print("Output checkpoints:", OUTPUT_DIR)
print("Final adapter:", FINAL_ADAPTER_DIR)
print("Training summary:", TRAINING_SUMMARY_PATH)
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Resuming the new continuation run from: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-7845


Step,Training Loss,Validation Loss



Training completed.
Output checkpoints: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1
Final adapter: /home/mabdallah/alexandriax_mt_14d/models/final_adapters/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1
Training summary: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/training_summary.json
TrainOutput(global_step=7845, training_loss=0.0, metrics={'train_runtime': 0.0027, 'train_samples_per_second': 22985899.322, 'train_steps_per_second': 2873237.415, 'total_flos': 4.9351531417790054e+17, 'train_loss': 0.0, 'epoch': 3.0})


In [9]:
# ============================================================
# Cell 9 — Inventory continuation checkpoints and eval losses
# ============================================================

def checkpoint_step(path):
    try:
        return int(path.name.replace("checkpoint-", ""))
    except Exception:
        return -1

checkpoint_dirs = sorted(
    [path for path in OUTPUT_DIR.glob("checkpoint-*") if path.is_dir()],
    key=checkpoint_step,
)

if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoints found under {OUTPUT_DIR}")

checkpoint_records = []

for checkpoint_dir in checkpoint_dirs:
    step = checkpoint_step(checkpoint_dir)
    trainer_state_path = checkpoint_dir / "trainer_state.json"

    global_step = None
    epoch = None
    latest_eval_loss = None
    latest_train_loss = None

    if trainer_state_path.exists():
        with open(trainer_state_path, "r", encoding="utf-8") as file:
            state = json.load(file)

        global_step = state.get("global_step")
        epoch = state.get("epoch")
        log_history = state.get("log_history", [])

        eval_entries = [
            entry
            for entry in log_history
            if "eval_loss" in entry and int(entry.get("step", -1)) <= step
        ]

        train_entries = [
            entry
            for entry in log_history
            if "loss" in entry and int(entry.get("step", -1)) <= step
        ]

        if eval_entries:
            latest_eval_loss = eval_entries[-1]["eval_loss"]

        if train_entries:
            latest_train_loss = train_entries[-1]["loss"]

    adapter_weight = checkpoint_dir / "adapter_model.safetensors"

    if not adapter_weight.exists():
        adapter_weight = checkpoint_dir / "adapter_model.bin"

    checkpoint_records.append(
        {
            "step": step,
            "checkpoint": checkpoint_dir.name,
            "path": str(checkpoint_dir),
            "global_step": global_step,
            "epoch": epoch,
            "latest_train_loss": latest_train_loss,
            "latest_eval_loss": latest_eval_loss,
            "has_adapter_config": (checkpoint_dir / "adapter_config.json").exists(),
            "has_adapter_weights": adapter_weight.exists(),
            "has_optimizer": (checkpoint_dir / "optimizer.pt").exists(),
            "has_scheduler": (checkpoint_dir / "scheduler.pt").exists(),
            "has_trainer_state": trainer_state_path.exists(),
        }
    )

checkpoint_inventory_df = pd.DataFrame(checkpoint_records).sort_values("step").reset_index(drop=True)

if not checkpoint_inventory_df["has_adapter_weights"].all():
    display(checkpoint_inventory_df[~checkpoint_inventory_df["has_adapter_weights"]])
    raise RuntimeError("At least one checkpoint has no adapter weights.")

CHECKPOINT_INVENTORY_PATH = DATA_RUN_DIR / "continuation_checkpoint_inventory.csv"
checkpoint_inventory_df.to_csv(
    CHECKPOINT_INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Parent checkpoint:", PARENT_CHECKPOINT_DIR)
print("Continuation checkpoints retained:", len(checkpoint_inventory_df))
print("First retained step:", int(checkpoint_inventory_df["step"].min()))
print("Latest retained step:", int(checkpoint_inventory_df["step"].max()))
print("Inventory:", CHECKPOINT_INVENTORY_PATH)

display(checkpoint_inventory_df)

Parent checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Continuation checkpoints retained: 79
First retained step: 100
Latest retained step: 7845
Inventory: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/continuation_checkpoint_inventory.csv


,step,checkpoint,path,global_step,epoch,latest_train_loss,latest_eval_loss,has_adapter_config,has_adapter_weights,has_optimizer,has_scheduler,has_trainer_state
0,100,checkpoint-100,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,100,0.038241,1.2981,1.312001,True,True,True,True,True
1,200,checkpoint-200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,200,0.076482,1.2175,1.278230,True,True,True,True,True
2,300,checkpoint-300,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,300,0.114723,1.2743,1.260493,True,True,True,True,True
3,400,checkpoint-400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,400,0.152964,1.1005,1.250430,True,True,True,True,True
4,500,checkpoint-500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,500,0.191205,1.1539,1.242694,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
74,7500,checkpoint-7500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,7500,2.868069,0.9987,1.195161,True,True,True,True,True
75,7600,checkpoint-7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,7600,2.906310,0.9385,1.195199,True,True,True,True,True
76,7700,checkpoint-7700,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,7700,2.944551,1.0249,1.195016,True,True,True,True,True
77,7800,checkpoint-7800,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,7800,2.982792,1.0175,1.194899,True,True,True,True,True


### **Prepare Beam-4 checkpoint candidates**

In [10]:
# ============================================================
# Cell 10 — Prepare coarse Beam-4 checkpoint candidates
# ============================================================

import gc
import hashlib
import json
import os
import time
from pathlib import Path

import pandas as pd
import torch

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

for object_name in ["trainer", "model", "base_model"]:
    if object_name in globals():
        del globals()[object_name]

gc.collect()
torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):
    torch.cuda.ipc_collect()

CHECKPOINT_SELECTION_DIR = (
    PROJECT_DIR
    / "checkpoint_selection"
    / f"{NEW_EXPERIMENT_NAME}_beam4_training_style"
)

SELECTION_PREDICTIONS_DIR = CHECKPOINT_SELECTION_DIR / "predictions"
SELECTION_RESULTS_DIR = CHECKPOINT_SELECTION_DIR / "results"

for path in [
    CHECKPOINT_SELECTION_DIR,
    SELECTION_PREDICTIONS_DIR,
    SELECTION_RESULTS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

COARSE_INTERVAL = 400
GEN_BATCH_SIZE = 2
SAVE_EVERY_GENERATED_ROWS = 50
MAX_NEW_TOKENS = 120

GENERATION_KWARGS = {
    "do_sample": False,
    "num_beams": 4,
    "num_return_sequences": 1,
    "length_penalty": 1.0,
    "early_stopping": True,
    "repetition_penalty": 1.05,
    "use_cache": True,
}

DECODE_TAG = "beam4"
PROMPT_FORMAT_TAG = "training_style_complete2shot_v1"

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
    use_fast=True,
    extra_special_tokens={},
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

def checkpoint_step(path):
    try:
        return int(Path(path).name.replace("checkpoint-", ""))
    except Exception:
        return -1

def has_adapter_weights(path):
    path = Path(path)

    return (
        (path / "adapter_model.safetensors").exists()
        or (path / "adapter_model.bin").exists()
    )

periodic_checkpoint_paths = sorted(
    [
        path
        for path in OUTPUT_DIR.glob("checkpoint-*")
        if path.is_dir()
        and (path / "adapter_config.json").exists()
        and has_adapter_weights(path)
    ],
    key=checkpoint_step,
)

if not periodic_checkpoint_paths:
    raise RuntimeError(f"No continuation checkpoints found under {OUTPUT_DIR}")

PERIODIC_CHECKPOINT_MAP = {
    checkpoint_step(path): path
    for path in periodic_checkpoint_paths
}

root_trainer_state_path = OUTPUT_DIR / "trainer_state.json"
final_continuation_step = max(PERIODIC_CHECKPOINT_MAP)

if root_trainer_state_path.exists():
    with open(root_trainer_state_path, "r", encoding="utf-8") as file:
        root_trainer_state = json.load(file)

    final_continuation_step = int(
        root_trainer_state.get(
            "global_step",
            final_continuation_step,
        )
    )

candidate_records = []
candidate_paths_seen = set()

def add_candidate(candidate_name, continuation_step, checkpoint_path, candidate_type):
    checkpoint_path = Path(checkpoint_path)
    resolved_path = str(checkpoint_path.resolve())

    if resolved_path in candidate_paths_seen:
        return

    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)

    if not (checkpoint_path / "adapter_config.json").exists():
        raise FileNotFoundError(
            f"adapter_config.json is missing from {checkpoint_path}"
        )

    if not has_adapter_weights(checkpoint_path):
        raise FileNotFoundError(
            f"Adapter weights are missing from {checkpoint_path}"
        )

    candidate_paths_seen.add(resolved_path)

    candidate_records.append(
        {
            "candidate_name": candidate_name,
            "continuation_step": int(continuation_step),
            "checkpoint_path": str(checkpoint_path),
            "candidate_type": candidate_type,
        }
    )

add_candidate(
    candidate_name="parent_checkpoint_16600",
    continuation_step=0,
    checkpoint_path=PARENT_CHECKPOINT_DIR,
    candidate_type="parent",
)

for step, checkpoint_path in sorted(PERIODIC_CHECKPOINT_MAP.items()):
    if step % COARSE_INTERVAL == 0:
        add_candidate(
            candidate_name=f"devft_step_{step:05d}",
            continuation_step=step,
            checkpoint_path=checkpoint_path,
            candidate_type="periodic",
        )

latest_periodic_step = max(PERIODIC_CHECKPOINT_MAP)

add_candidate(
    candidate_name=f"devft_step_{latest_periodic_step:05d}",
    continuation_step=latest_periodic_step,
    checkpoint_path=PERIODIC_CHECKPOINT_MAP[latest_periodic_step],
    candidate_type="latest_periodic",
)

if (
    FINAL_ADAPTER_DIR.exists()
    and (FINAL_ADAPTER_DIR / "adapter_config.json").exists()
    and has_adapter_weights(FINAL_ADAPTER_DIR)
):
    add_candidate(
        candidate_name=f"devft_final_{final_continuation_step:05d}",
        continuation_step=final_continuation_step,
        checkpoint_path=FINAL_ADAPTER_DIR,
        candidate_type="final_adapter",
    )

COARSE_CANDIDATE_DF = (
    pd.DataFrame(candidate_records)
    .sort_values(
        ["continuation_step", "candidate_type"],
        kind="stable",
    )
    .reset_index(drop=True)
)

COARSE_CANDIDATE_PATH = (
    CHECKPOINT_SELECTION_DIR
    / "coarse_checkpoint_candidates.csv"
)

COARSE_CANDIDATE_DF.to_csv(
    COARSE_CANDIDATE_PATH,
    index=False,
    encoding="utf-8-sig",
)

def message_content(messages, role):
    for message in messages:
        if message.get("role") == role:
            return str(message.get("content", ""))
    return ""

def make_inference_text(messages):
    system_text = message_content(messages, "system").strip()
    user_text = message_content(messages, "user").strip()

    return (
        f"{SYSTEM_MARKER}\n"
        f"{system_text}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text}\n\n"
        f"{RESPONSE_MARKER}\n"
    )

selection_inference_df = selection_prompt_df[
    [
        "source_id",
        "config",
        "conversation_id",
        "turn_order",
        "source_text",
        "target_arabic",
        "messages",
    ]
].copy()

selection_inference_df["source_id"] = (
    selection_inference_df["source_id"].astype(str)
)

selection_inference_df["reference_arabic"] = (
    selection_inference_df["target_arabic"]
    .fillna("")
    .astype(str)
    .str.strip()
)

selection_inference_df["inference_prompt"] = [
    make_inference_text(messages)
    for messages in tqdm(
        selection_inference_df["messages"],
        total=len(selection_inference_df),
        desc="Building selection prompts",
    )
]

selection_inference_df = selection_inference_df.drop(
    columns=["messages"]
)

selection_inference_df["selection_order"] = range(
    len(selection_inference_df)
)

selection_hasher = hashlib.sha256()

for row in selection_inference_df.itertuples(index=False):
    selection_hasher.update(str(row.source_id).encode("utf-8"))
    selection_hasher.update(b"\n")
    selection_hasher.update(str(row.inference_prompt).encode("utf-8"))
    selection_hasher.update(b"\n")

SELECTION_FINGERPRINT = selection_hasher.hexdigest()

SELECTION_INFERENCE_CACHE = (
    CHECKPOINT_SELECTION_DIR
    / "selection_inference_rows.pkl"
)

selection_inference_df.to_pickle(
    SELECTION_INFERENCE_CACHE
)

assert len(selection_inference_df) == len(selection_df)
assert selection_inference_df["source_id"].nunique() == len(selection_df)
assert selection_inference_df["reference_arabic"].ne("").all()
assert selection_inference_df["config"].nunique() == 13

print("Selection rows:", len(selection_inference_df))
print("Selection countries:", sorted(selection_inference_df["config"].unique()))
print("Selection fingerprint:", SELECTION_FINGERPRINT)
print("Coarse candidates:", len(COARSE_CANDIDATE_DF))
print("Coarse interval:", COARSE_INTERVAL)
print("Generation:", GENERATION_KWARGS)

display(COARSE_CANDIDATE_DF)

Building selection prompts:   0%|          | 0/5772 [00:00<?, ?it/s]

Selection rows: 5772
Selection countries: ['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']
Selection fingerprint: 3764b1fa7f58ddd0b4c2ba88fb79784d212ec3c0851cedbbe400ce2c850e46ae
Coarse candidates: 22
Coarse interval: 400
Generation: {'do_sample': False, 'num_beams': 4, 'num_return_sequences': 1, 'length_penalty': 1.0, 'early_stopping': True, 'repetition_penalty': 1.05, 'use_cache': True}


,candidate_name,continuation_step,checkpoint_path,candidate_type
0,parent_checkpoint_16600,0,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,parent
1,devft_step_00400,400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic
2,devft_step_00800,800,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic
3,devft_step_01200,1200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic
4,devft_step_01600,1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic
5,devft_step_02000,2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic
6,devft_step_02400,2400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic
7,devft_step_02800,2800,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic
8,devft_step_03200,3200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic
9,devft_step_03600,3600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,periodic


In [11]:
# ============================================================
# Cell 11 — Resumable Beam-4 generation and scoring functions
# ============================================================

import sacrebleu

def clean_generated_translation(text):
    text = str(text).strip()

    if RESPONSE_MARKER in text:
        text = text.split(RESPONSE_MARKER)[-1].strip()

    stop_markers = [
        SYSTEM_MARKER,
        INSTRUCTION_MARKER,
        "### System",
        "### Instruction",
        "### English",
        "Task:",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    text = text.replace("<|endoftext|>", "").strip()
    text = text.replace("<s>", "").replace("</s>", "").strip()

    return text

def load_checkpoint_for_inference(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_DIR,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
        local_files_only=True,
        attn_implementation="sdpa",
    )

    inference_model = PeftModel.from_pretrained(
        base_model,
        checkpoint_path,
        is_trainable=False,
        local_files_only=True,
    )

    inference_model.eval()
    inference_model.config.use_cache = True
    inference_model.config.pad_token_id = tokenizer.pad_token_id

    return base_model, inference_model

def generate_prompt_batch(inference_model, prompts):
    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )

    model_device = next(inference_model.parameters()).device

    encoded = {
        key: value.to(model_device)
        for key, value in encoded.items()
        if key in {"input_ids", "attention_mask"}
    }

    input_length = encoded["input_ids"].shape[1]

    with torch.inference_mode():
        generated = inference_model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **GENERATION_KWARGS,
        )

    generated_tokens = generated[:, input_length:]

    return tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
    )

def generate_rows_safely(inference_model, rows):
    prompts = [
        str(row["inference_prompt"])
        for row in rows
    ]

    try:
        raw_predictions = generate_prompt_batch(
            inference_model,
            prompts,
        )

        return [
            {
                "raw_prediction": raw_prediction,
                "clean_prediction": clean_generated_translation(
                    raw_prediction
                ),
                "generation_error": "",
            }
            for raw_prediction in raw_predictions
        ]

    except Exception as batch_error:
        print("Batch generation failed:", repr(batch_error))
        print("Retrying rows individually.")

        torch.cuda.empty_cache()
        results = []

        for row in rows:
            try:
                raw_prediction = generate_prompt_batch(
                    inference_model,
                    [str(row["inference_prompt"])],
                )[0]

                results.append(
                    {
                        "raw_prediction": raw_prediction,
                        "clean_prediction": clean_generated_translation(
                            raw_prediction
                        ),
                        "generation_error": "",
                    }
                )

            except Exception as row_error:
                results.append(
                    {
                        "raw_prediction": "",
                        "clean_prediction": "",
                        "generation_error": repr(row_error),
                    }
                )

                torch.cuda.empty_cache()

        return results

def atomic_save_prediction_records(records_by_id, prediction_path):
    frame = pd.DataFrame(
        list(records_by_id.values())
    )

    order_frame = selection_inference_df[
        ["source_id", "selection_order"]
    ].copy()

    frame["source_id"] = frame["source_id"].astype(str)

    frame = (
        frame.merge(
            order_frame,
            on="source_id",
            how="left",
        )
        .sort_values("selection_order")
        .drop(columns=["selection_order"])
        .reset_index(drop=True)
    )

    temporary_path = Path(
        str(prediction_path) + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig",
    )

    os.replace(
        temporary_path,
        prediction_path,
    )

    return frame

def compute_checkpoint_metrics(prediction_df):
    country_rows = []

    for country in sorted(
        prediction_df["config"].astype(str).unique()
    ):
        country_df = prediction_df[
            prediction_df["config"].astype(str) == country
        ].copy()

        predictions = (
            country_df["clean_prediction"]
            .fillna("")
            .astype(str)
            .tolist()
        )

        references = (
            country_df["reference_arabic"]
            .fillna("")
            .astype(str)
            .tolist()
        )

        country_rows.append(
            {
                "country": country,
                "turns": len(country_df),
                "spBLEU": float(
                    sacrebleu.corpus_bleu(
                        predictions,
                        [references],
                        tokenize="flores200",
                    ).score
                ),
                "chrF++": float(
                    sacrebleu.corpus_chrf(
                        predictions,
                        [references],
                        word_order=2,
                    ).score
                ),
                "BLEU": float(
                    sacrebleu.corpus_bleu(
                        predictions,
                        [references],
                    ).score
                ),
                "chrF": float(
                    sacrebleu.corpus_chrf(
                        predictions,
                        [references],
                    ).score
                ),
            }
        )

    country_metric_df = pd.DataFrame(
        country_rows
    ).sort_values("country").reset_index(drop=True)

    all_predictions = (
        prediction_df["clean_prediction"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    all_references = (
        prediction_df["reference_arabic"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    summary = {
        "selection_turns": int(len(prediction_df)),
        "selection_countries": int(
            country_metric_df["country"].nunique()
        ),
        "macro_spBLEU": float(
            country_metric_df["spBLEU"].mean()
        ),
        "macro_chrF++": float(
            country_metric_df["chrF++"].mean()
        ),
        "corpus_spBLEU": float(
            sacrebleu.corpus_bleu(
                all_predictions,
                [all_references],
                tokenize="flores200",
            ).score
        ),
        "corpus_chrF++": float(
            sacrebleu.corpus_chrf(
                all_predictions,
                [all_references],
                word_order=2,
            ).score
        ),
    }

    return summary, country_metric_df

def candidate_fingerprint(candidate):
    payload = {
        "candidate_name": str(candidate["candidate_name"]),
        "continuation_step": int(candidate["continuation_step"]),
        "checkpoint_path": str(candidate["checkpoint_path"]),
        "selection_fingerprint": SELECTION_FINGERPRINT,
        "prompt_format": PROMPT_FORMAT_TAG,
        "decode": DECODE_TAG,
        "max_seq_length": MAX_SEQ_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "generation_kwargs": GENERATION_KWARGS,
    }

    fingerprint = hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()

    return payload, fingerprint

def run_checkpoint_candidate(candidate):
    candidate = dict(candidate)
    candidate_name = str(candidate["candidate_name"])
    checkpoint_path = Path(candidate["checkpoint_path"])

    candidate_dir = (
        SELECTION_PREDICTIONS_DIR
        / candidate_name
    )

    candidate_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    prediction_path = (
        candidate_dir
        / "selection_predictions.csv"
    )

    manifest_path = (
        candidate_dir
        / "candidate_manifest.json"
    )

    per_country_path = (
        candidate_dir
        / "per_country_metrics.csv"
    )

    summary_path = (
        candidate_dir
        / "checkpoint_summary.json"
    )

    manifest_payload, fingerprint = (
        candidate_fingerprint(candidate)
    )

    manifest_payload["fingerprint"] = fingerprint

    if manifest_path.exists():
        with open(
            manifest_path,
            "r",
            encoding="utf-8",
        ) as file:
            existing_manifest = json.load(file)

        if (
            existing_manifest.get("fingerprint")
            != fingerprint
        ):
            raise RuntimeError(
                f"Fingerprint mismatch for {candidate_name}. "
                "Use a fresh candidate folder."
            )
    else:
        with open(
            manifest_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                manifest_payload,
                file,
                ensure_ascii=False,
                indent=2,
            )

    records_by_id = {}

    if prediction_path.exists():
        existing_predictions = pd.read_csv(
            prediction_path
        )

        existing_predictions["source_id"] = (
            existing_predictions["source_id"]
            .astype(str)
        )

        existing_predictions["clean_prediction"] = (
            existing_predictions["clean_prediction"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        existing_predictions["generation_error"] = (
            existing_predictions["generation_error"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        expected_ids = set(
            selection_inference_df["source_id"]
            .astype(str)
        )

        existing_predictions = (
            existing_predictions[
                existing_predictions["source_id"].isin(
                    expected_ids
                )
            ]
            .drop_duplicates(
                subset=["source_id"],
                keep="last",
            )
        )

        records_by_id = {
            str(row["source_id"]): row
            for row in existing_predictions.to_dict(
                "records"
            )
        }

    completed_ids = {
        source_id
        for source_id, record in records_by_id.items()
        if str(
            record.get("clean_prediction", "")
        ).strip()
        and not str(
            record.get("generation_error", "")
        ).strip()
    }

    pending_rows = [
        row
        for row in selection_inference_df.to_dict(
            "records"
        )
        if str(row["source_id"]) not in completed_ids
    ]

    print("\n" + "=" * 90)
    print("CHECKPOINT CANDIDATE:", candidate_name)
    print("Continuation step:", candidate["continuation_step"])
    print("Checkpoint:", checkpoint_path)
    print("Completed:", len(completed_ids))
    print("Pending:", len(pending_rows))
    print("=" * 90)

    base_model = None
    inference_model = None

    try:
        if pending_rows:
            base_model, inference_model = (
                load_checkpoint_for_inference(
                    checkpoint_path
                )
            )

            generated_since_save = 0

            for start in tqdm(
                range(
                    0,
                    len(pending_rows),
                    GEN_BATCH_SIZE,
                ),
                desc=candidate_name,
            ):
                batch_rows = pending_rows[
                    start : start + GEN_BATCH_SIZE
                ]

                batch_results = generate_rows_safely(
                    inference_model,
                    batch_rows,
                )

                for row, result in zip(
                    batch_rows,
                    batch_results,
                ):
                    source_id = str(
                        row["source_id"]
                    )

                    records_by_id[source_id] = {
                        "source_id": source_id,
                        "config": row["config"],
                        "conversation_id": row[
                            "conversation_id"
                        ],
                        "turn_order": int(
                            row["turn_order"]
                        ),
                        "source_text": row[
                            "source_text"
                        ],
                        "reference_arabic": row[
                            "reference_arabic"
                        ],
                        "raw_prediction": result[
                            "raw_prediction"
                        ],
                        "clean_prediction": result[
                            "clean_prediction"
                        ],
                        "generation_error": result[
                            "generation_error"
                        ],
                        "candidate_name": candidate_name,
                        "continuation_step": int(
                            candidate[
                                "continuation_step"
                            ]
                        ),
                        "checkpoint_path": str(
                            checkpoint_path
                        ),
                        "generated_at": time.strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                    }

                    generated_since_save += 1

                if (
                    generated_since_save
                    >= SAVE_EVERY_GENERATED_ROWS
                ):
                    saved_frame = (
                        atomic_save_prediction_records(
                            records_by_id,
                            prediction_path,
                        )
                    )

                    print(
                        "Saved:",
                        len(saved_frame),
                        "/",
                        len(selection_inference_df),
                    )

                    generated_since_save = 0

            atomic_save_prediction_records(
                records_by_id,
                prediction_path,
            )

    finally:
        if inference_model is not None:
            del inference_model

        if base_model is not None:
            del base_model

        gc.collect()
        torch.cuda.empty_cache()

        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()

    prediction_df = pd.read_csv(
        prediction_path
    )

    prediction_df["source_id"] = (
        prediction_df["source_id"].astype(str)
    )

    prediction_df["clean_prediction"] = (
        prediction_df["clean_prediction"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    prediction_df["generation_error"] = (
        prediction_df["generation_error"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    expected_ids = set(
        selection_inference_df["source_id"]
        .astype(str)
    )

    actual_ids = set(
        prediction_df["source_id"]
        .astype(str)
    )

    missing_ids = expected_ids - actual_ids
    extra_ids = actual_ids - expected_ids
    duplicate_count = int(
        prediction_df["source_id"]
        .duplicated()
        .sum()
    )

    empty_count = int(
        prediction_df["clean_prediction"]
        .eq("")
        .sum()
    )

    error_count = int(
        prediction_df["generation_error"]
        .ne("")
        .sum()
    )

    print("Missing:", len(missing_ids))
    print("Extra:", len(extra_ids))
    print("Duplicates:", duplicate_count)
    print("Empty:", empty_count)
    print("Errors:", error_count)

    if (
        missing_ids
        or extra_ids
        or duplicate_count
        or empty_count
        or error_count
        or len(prediction_df)
        != len(selection_inference_df)
    ):
        raise RuntimeError(
            f"Candidate {candidate_name} is incomplete. "
            "Rerun this cell to retry missing/error rows."
        )

    summary, country_metric_df = (
        compute_checkpoint_metrics(
            prediction_df
        )
    )

    summary.update(
        {
            "candidate_name": candidate_name,
            "candidate_type": candidate[
                "candidate_type"
            ],
            "continuation_step": int(
                candidate[
                    "continuation_step"
                ]
            ),
            "checkpoint_path": str(
                checkpoint_path
            ),
            "prediction_path": str(
                prediction_path
            ),
            "per_country_path": str(
                per_country_path
            ),
            "candidate_dir": str(
                candidate_dir
            ),
            "fingerprint": fingerprint,
        }
    )

    country_metric_df[
        "candidate_name"
    ] = candidate_name

    country_metric_df[
        "continuation_step"
    ] = int(
        candidate["continuation_step"]
    )

    country_metric_df[
        "checkpoint_path"
    ] = str(checkpoint_path)

    country_metric_df.to_csv(
        per_country_path,
        index=False,
        encoding="utf-8-sig",
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("\nMacro spBLEU:", f"{summary['macro_spBLEU']:.6f}")
    print("Macro chrF++:", f"{summary['macro_chrF++']:.6f}")

    return summary

### **Beam4 Sweep**

In [12]:
# ============================================================
# Cell 12 — Run or resume the coarse Beam-4 sweep
# ============================================================

COARSE_RESULTS_LIVE_PATH = (
    SELECTION_RESULTS_DIR
    / "coarse_checkpoint_results_live.csv"
)

coarse_results = []

for candidate in COARSE_CANDIDATE_DF.to_dict(
    "records"
):
    result = run_checkpoint_candidate(
        candidate
    )

    coarse_results.append(result)

    coarse_result_df = (
        pd.DataFrame(coarse_results)
        .sort_values(
            [
                "macro_spBLEU",
                "macro_chrF++",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    coarse_result_df.to_csv(
        COARSE_RESULTS_LIVE_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    display(
        coarse_result_df[
            [
                "candidate_name",
                "candidate_type",
                "continuation_step",
                "macro_spBLEU",
                "macro_chrF++",
            ]
        ]
    )

print("\nCoarse Beam-4 sweep completed.")
print("Results:", COARSE_RESULTS_LIVE_PATH)


CHECKPOINT CANDIDATE: parent_checkpoint_16600
Continuation step: 0
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Completed: 5772
Pending: 0
Missing: 0
Extra: 0
Duplicates: 0
Empty: 0
Errors: 0

Macro spBLEU: 27.037369
Macro chrF++: 42.901217


,candidate_name,candidate_type,continuation_step,macro_spBLEU,macro_chrF++
0,parent_checkpoint_16600,parent,0,27.037369,42.901217



CHECKPOINT CANDIDATE: devft_step_00400
Continuation step: 400
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-400
Completed: 5772
Pending: 0
Missing: 0
Extra: 0
Duplicates: 0
Empty: 0
Errors: 0

Macro spBLEU: 28.717941
Macro chrF++: 43.658772


,candidate_name,candidate_type,continuation_step,macro_spBLEU,macro_chrF++
0,devft_step_00400,periodic,400,28.717941,43.658772
1,parent_checkpoint_16600,parent,0,27.037369,42.901217



CHECKPOINT CANDIDATE: devft_step_00800
Continuation step: 800
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-800
Completed: 5772
Pending: 0
Missing: 0
Extra: 0
Duplicates: 0
Empty: 0
Errors: 0

Macro spBLEU: 28.977036
Macro chrF++: 43.944473


...


CHECKPOINT CANDIDATE: devft_final_07845
Continuation step: 7845
Checkpoint: /home/mabdallah/alexandriax_mt_14d/models/final_adapters/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1
Completed: 5772
Pending: 0
Missing: 0
Extra: 0
Duplicates: 0
Empty: 0
Errors: 0

Macro spBLEU: 29.848937
Macro chrF++: 44.588605


,candidate_name,candidate_type,continuation_step,macro_spBLEU,macro_chrF++
0,devft_step_05200,periodic,5200,29.942952,44.634356
1,devft_final_07845,final_adapter,7845,29.848937,44.588605
2,devft_step_06400,periodic,6400,29.839469,44.571395
3,devft_step_05600,periodic,5600,29.834589,44.552618
4,devft_step_06800,periodic,6800,29.821187,44.585376
5,devft_step_07600,periodic,7600,29.799508,44.543234
6,devft_step_07200,periodic,7200,29.798787,44.527760
7,devft_step_06000,periodic,6000,29.778923,44.516236
8,devft_step_03600,periodic,3600,29.759006,44.545702
9,devft_step_04800,periodic,4800,29.741686,44.516227



CHECKPOINT CANDIDATE: devft_step_07845
Continuation step: 7845
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-7845
Completed: 5772
Pending: 0
Missing: 0
Extra: 0
Duplicates: 0
Empty: 0
Errors: 0

Macro spBLEU: 29.848937
Macro chrF++: 44.588605


,candidate_name,candidate_type,continuation_step,macro_spBLEU,macro_chrF++
0,devft_step_05200,periodic,5200,29.942952,44.634356
1,devft_final_07845,final_adapter,7845,29.848937,44.588605
2,devft_step_07845,latest_periodic,7845,29.848937,44.588605
3,devft_step_06400,periodic,6400,29.839469,44.571395
4,devft_step_05600,periodic,5600,29.834589,44.552618
5,devft_step_06800,periodic,6800,29.821187,44.585376
6,devft_step_07600,periodic,7600,29.799508,44.543234
7,devft_step_07200,periodic,7200,29.798787,44.527760
8,devft_step_06000,periodic,6000,29.778923,44.516236
9,devft_step_03600,periodic,3600,29.759006,44.545702



Coarse Beam-4 sweep completed.
Results: /home/mabdallah/alexandriax_mt_14d/checkpoint_selection/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1_beam4_training_style/results/coarse_checkpoint_results_live.csv


### Ranking

In [13]:
# ============================================================
# Cell 13 — Rank coarse checkpoints against parent 16600
# ============================================================

coarse_ranked_df = pd.read_csv(
    COARSE_RESULTS_LIVE_PATH
)

coarse_ranked_df = (
    coarse_ranked_df
    .sort_values(
        [
            "macro_spBLEU",
            "macro_chrF++",
            "continuation_step",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

coarse_ranked_df[
    "coarse_rank"
] = range(
    1,
    len(coarse_ranked_df) + 1,
)

parent_rows = coarse_ranked_df[
    coarse_ranked_df[
        "candidate_name"
    ] == "parent_checkpoint_16600"
]

if len(parent_rows) != 1:
    raise RuntimeError(
        "Parent checkpoint result is missing or duplicated."
    )

parent_spbleu = float(
    parent_rows.iloc[0]["macro_spBLEU"]
)

parent_chrfpp = float(
    parent_rows.iloc[0]["macro_chrF++"]
)

coarse_ranked_df[
    "spBLEU_delta_vs_parent"
] = (
    coarse_ranked_df["macro_spBLEU"]
    - parent_spbleu
)

coarse_ranked_df[
    "chrF++_delta_vs_parent"
] = (
    coarse_ranked_df["macro_chrF++"]
    - parent_chrfpp
)

COARSE_RANKED_PATH = (
    SELECTION_RESULTS_DIR
    / "coarse_checkpoint_results_ranked.csv"
)

coarse_ranked_df.to_csv(
    COARSE_RANKED_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Parent checkpoint macro spBLEU:", parent_spbleu)
print("Parent checkpoint macro chrF++:", parent_chrfpp)

display(
    coarse_ranked_df[
        [
            "coarse_rank",
            "candidate_name",
            "continuation_step",
            "macro_spBLEU",
            "spBLEU_delta_vs_parent",
            "macro_chrF++",
            "chrF++_delta_vs_parent",
        ]
    ]
)

BEST_COARSE_CANDIDATE = (
    coarse_ranked_df.iloc[0].to_dict()
)

print("\nBest coarse candidate:")
print("Name:", BEST_COARSE_CANDIDATE["candidate_name"])
print("Step:", int(BEST_COARSE_CANDIDATE["continuation_step"]))
print("Macro spBLEU:", BEST_COARSE_CANDIDATE["macro_spBLEU"])
print("Macro chrF++:", BEST_COARSE_CANDIDATE["macro_chrF++"])
print("Path:", BEST_COARSE_CANDIDATE["checkpoint_path"])

Parent checkpoint macro spBLEU: 27.037368946287664
Parent checkpoint macro chrF++: 42.90121737014235


,coarse_rank,candidate_name,continuation_step,macro_spBLEU,spBLEU_delta_vs_parent,macro_chrF++,chrF++_delta_vs_parent
0,1,devft_step_05200,5200,29.942952,2.905583,44.634356,1.733139
1,2,devft_final_07845,7845,29.848937,2.811568,44.588605,1.687388
2,3,devft_step_07845,7845,29.848937,2.811568,44.588605,1.687388
3,4,devft_step_06400,6400,29.839469,2.802100,44.571395,1.670178
4,5,devft_step_05600,5600,29.834589,2.797220,44.552618,1.651400
5,6,devft_step_06800,6800,29.821187,2.783818,44.585376,1.684159
6,7,devft_step_07600,7600,29.799508,2.762139,44.543234,1.642017
7,8,devft_step_07200,7200,29.798787,2.761418,44.527760,1.626543
8,9,devft_step_06000,6000,29.778923,2.741554,44.516236,1.615018
9,10,devft_step_03600,3600,29.759006,2.721637,44.545702,1.644485



Best coarse candidate:
Name: devft_step_05200
Step: 5200
Macro spBLEU: 29.942952322676767
Macro chrF++: 44.634356315766986
Path: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-5200


In [15]:
# # ============================================================
# # Cell 14 — Evaluate ±300 steps around top coarse regions
# # ============================================================

# REFINEMENT_RADIUS = 300
# TOP_COARSE_REGIONS = 3

# top_new_coarse_df = (
#     coarse_ranked_df[
#         coarse_ranked_df[
#             "continuation_step"
#         ] > 0
#     ]
#     .head(TOP_COARSE_REGIONS)
#     .copy()
# )

# refinement_center_steps = sorted(
#     top_new_coarse_df[
#         "continuation_step"
#     ]
#     .astype(int)
#     .unique()
#     .tolist()
# )

# already_evaluated_paths = set(
#     coarse_ranked_df[
#         "checkpoint_path"
#     ]
#     .astype(str)
#     .map(
#         lambda path: str(
#             Path(path).resolve()
#         )
#     )
# )

# refinement_records = []

# for step, checkpoint_path in sorted(
#     PERIODIC_CHECKPOINT_MAP.items()
# ):
#     if not any(
#         abs(step - center_step)
#         <= REFINEMENT_RADIUS
#         for center_step in refinement_center_steps
#     ):
#         continue

#     resolved_path = str(
#         checkpoint_path.resolve()
#     )

#     if resolved_path in already_evaluated_paths:
#         continue

#     refinement_records.append(
#         {
#             "candidate_name": f"devft_step_{step:05d}",
#             "candidate_type": "refinement",
#             "continuation_step": int(step),
#             "checkpoint_path": str(checkpoint_path),
#         }
#     )

# REFINEMENT_CANDIDATE_DF = (
#     pd.DataFrame(refinement_records)
#     if refinement_records
#     else pd.DataFrame(
#         columns=[
#             "candidate_name",
#             "candidate_type",
#             "continuation_step",
#             "checkpoint_path",
#         ]
#     )
# )

# if len(REFINEMENT_CANDIDATE_DF):
#     REFINEMENT_CANDIDATE_DF = (
#         REFINEMENT_CANDIDATE_DF
#         .sort_values("continuation_step")
#         .reset_index(drop=True)
#     )

# print("Top coarse region centers:", refinement_center_steps)
# print("Additional refinement checkpoints:", len(REFINEMENT_CANDIDATE_DF))

# display(REFINEMENT_CANDIDATE_DF)

# REFINEMENT_RESULTS_LIVE_PATH = (
#     SELECTION_RESULTS_DIR
#     / "refinement_checkpoint_results_live.csv"
# )

# refinement_results = []

# for candidate in REFINEMENT_CANDIDATE_DF.to_dict(
#     "records"
# ):
#     result = run_checkpoint_candidate(
#         candidate
#     )

#     refinement_results.append(result)

#     refinement_result_df = (
#         pd.DataFrame(refinement_results)
#         .sort_values(
#             [
#                 "macro_spBLEU",
#                 "macro_chrF++",
#             ],
#             ascending=[
#                 False,
#                 False,
#             ],
#         )
#         .reset_index(drop=True)
#     )

#     refinement_result_df.to_csv(
#         REFINEMENT_RESULTS_LIVE_PATH,
#         index=False,
#         encoding="utf-8-sig",
#     )

#     display(
#         refinement_result_df[
#             [
#                 "candidate_name",
#                 "continuation_step",
#                 "macro_spBLEU",
#                 "macro_chrF++",
#             ]
#         ]
#     )

# print("\nRefinement sweep completed.")

In [16]:
# ============================================================
# Cell 15 — Final checkpoint ranking and per-country analysis
# ============================================================

completed_summaries = []

for summary_path in sorted(
    SELECTION_PREDICTIONS_DIR.glob(
        "*/checkpoint_summary.json"
    )
):
    with open(
        summary_path,
        "r",
        encoding="utf-8",
    ) as file:
        completed_summaries.append(
            json.load(file)
        )

if not completed_summaries:
    raise RuntimeError(
        "No completed checkpoint summaries found."
    )

final_checkpoint_ranked_df = (
    pd.DataFrame(completed_summaries)
    .drop_duplicates(
        subset=["candidate_name"],
        keep="last",
    )
    .sort_values(
        [
            "macro_spBLEU",
            "macro_chrF++",
            "continuation_step",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

final_checkpoint_ranked_df[
    "final_rank"
] = range(
    1,
    len(final_checkpoint_ranked_df) + 1,
)

parent_row = final_checkpoint_ranked_df[
    final_checkpoint_ranked_df[
        "candidate_name"
    ] == "parent_checkpoint_16600"
]

if len(parent_row) != 1:
    raise RuntimeError(
        "The completed parent checkpoint result is missing."
    )

parent_spbleu = float(
    parent_row.iloc[0]["macro_spBLEU"]
)

parent_chrfpp = float(
    parent_row.iloc[0]["macro_chrF++"]
)

final_checkpoint_ranked_df[
    "spBLEU_delta_vs_parent"
] = (
    final_checkpoint_ranked_df[
        "macro_spBLEU"
    ]
    - parent_spbleu
)

final_checkpoint_ranked_df[
    "chrF++_delta_vs_parent"
] = (
    final_checkpoint_ranked_df[
        "macro_chrF++"
    ]
    - parent_chrfpp
)

FINAL_CHECKPOINT_RANKING_PATH = (
    SELECTION_RESULTS_DIR
    / "final_checkpoint_ranking.csv"
)

final_checkpoint_ranked_df.to_csv(
    FINAL_CHECKPOINT_RANKING_PATH,
    index=False,
    encoding="utf-8-sig",
)

BEST_CHECKPOINT_ROW = (
    final_checkpoint_ranked_df.iloc[0].to_dict()
)

BEST_CHECKPOINT_CANDIDATE = str(
    BEST_CHECKPOINT_ROW[
        "candidate_name"
    ]
)

BEST_CONTINUATION_STEP = int(
    BEST_CHECKPOINT_ROW[
        "continuation_step"
    ]
)

BEST_CHECKPOINT_PATH = Path(
    BEST_CHECKPOINT_ROW[
        "checkpoint_path"
    ]
)

per_country_parts = []

for summary in completed_summaries:
    per_country_path = Path(
        summary["per_country_path"]
    )

    if not per_country_path.exists():
        continue

    country_frame = pd.read_csv(
        per_country_path
    )

    country_frame[
        "candidate_name"
    ] = summary["candidate_name"]

    country_frame[
        "candidate_type"
    ] = summary["candidate_type"]

    country_frame[
        "continuation_step"
    ] = int(
        summary["continuation_step"]
    )

    country_frame[
        "checkpoint_path"
    ] = summary["checkpoint_path"]

    per_country_parts.append(
        country_frame
    )

all_checkpoint_country_metrics_df = (
    pd.concat(
        per_country_parts,
        ignore_index=True,
    )
)

all_checkpoint_country_metrics_df = (
    all_checkpoint_country_metrics_df
    .sort_values(
        [
            "country",
            "spBLEU",
            "chrF++",
            "continuation_step",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

best_checkpoint_per_country_df = (
    all_checkpoint_country_metrics_df
    .groupby(
        "country",
        as_index=False,
        sort=True,
    )
    .first()
    .rename(
        columns={
            "candidate_name": "best_candidate",
            "continuation_step": "best_continuation_step",
            "checkpoint_path": "best_checkpoint_path",
            "spBLEU": "selection_spBLEU",
            "chrF++": "selection_chrF++",
        }
    )
)

PER_COUNTRY_CHECKPOINT_PATH = (
    SELECTION_RESULTS_DIR
    / "best_checkpoint_per_country.csv"
)

best_checkpoint_per_country_df.to_csv(
    PER_COUNTRY_CHECKPOINT_PATH,
    index=False,
    encoding="utf-8-sig",
)

checkpoint_mixture_macro_spbleu = float(
    best_checkpoint_per_country_df[
        "selection_spBLEU"
    ].mean()
)

checkpoint_mixture_macro_chrfpp = float(
    best_checkpoint_per_country_df[
        "selection_chrF++"
    ].mean()
)

best_checkpoint_payload = {
    "best_candidate": BEST_CHECKPOINT_CANDIDATE,
    "best_continuation_step": BEST_CONTINUATION_STEP,
    "best_checkpoint_path": str(BEST_CHECKPOINT_PATH),
    "macro_spBLEU": float(
        BEST_CHECKPOINT_ROW[
            "macro_spBLEU"
        ]
    ),
    "macro_chrF++": float(
        BEST_CHECKPOINT_ROW[
            "macro_chrF++"
        ]
    ),
    "spBLEU_delta_vs_parent": float(
        BEST_CHECKPOINT_ROW[
            "spBLEU_delta_vs_parent"
        ]
    ),
    "chrF++_delta_vs_parent": float(
        BEST_CHECKPOINT_ROW[
            "chrF++_delta_vs_parent"
        ]
    ),
    "checkpoint_mixture_macro_spBLEU": checkpoint_mixture_macro_spbleu,
    "checkpoint_mixture_macro_chrF++": checkpoint_mixture_macro_chrfpp,
    "selection_fingerprint": SELECTION_FINGERPRINT,
}

BEST_CHECKPOINT_JSON_PATH = (
    SELECTION_RESULTS_DIR
    / "best_checkpoint.json"
)

with open(
    BEST_CHECKPOINT_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_checkpoint_payload,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("=" * 90)
print("FINAL BEST CHECKPOINT")
print("=" * 90)
print("Candidate:", BEST_CHECKPOINT_CANDIDATE)
print("Continuation step:", BEST_CONTINUATION_STEP)
print("Checkpoint:", BEST_CHECKPOINT_PATH)
print(
    "Macro spBLEU:",
    f"{BEST_CHECKPOINT_ROW['macro_spBLEU']:.6f}",
)
print(
    "Delta versus checkpoint 16600:",
    f"{BEST_CHECKPOINT_ROW['spBLEU_delta_vs_parent']:+.6f}",
)
print(
    "Macro chrF++:",
    f"{BEST_CHECKPOINT_ROW['macro_chrF++']:.6f}",
)
print(
    "Potential per-country checkpoint mixture spBLEU:",
    f"{checkpoint_mixture_macro_spbleu:.6f}",
)
print(
    "Potential per-country checkpoint mixture chrF++:",
    f"{checkpoint_mixture_macro_chrfpp:.6f}",
)

print("\nGlobal checkpoint ranking:")
display(
    final_checkpoint_ranked_df[
        [
            "final_rank",
            "candidate_name",
            "candidate_type",
            "continuation_step",
            "macro_spBLEU",
            "spBLEU_delta_vs_parent",
            "macro_chrF++",
            "chrF++_delta_vs_parent",
        ]
    ]
)

print("\nBest checkpoint independently for each country:")
display(
    best_checkpoint_per_country_df[
        [
            "country",
            "best_candidate",
            "best_continuation_step",
            "selection_spBLEU",
            "selection_chrF++",
        ]
    ]
)

print("\nSaved global ranking:", FINAL_CHECKPOINT_RANKING_PATH)
print("Saved per-country ranking:", PER_COUNTRY_CHECKPOINT_PATH)
print("Saved best checkpoint:", BEST_CHECKPOINT_JSON_PATH)

FINAL BEST CHECKPOINT
Candidate: devft_step_05200
Continuation step: 5200
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-5200
Macro spBLEU: 29.942952
Delta versus checkpoint 16600: +2.905583
Macro chrF++: 44.634356
Potential per-country checkpoint mixture spBLEU: 30.184959
Potential per-country checkpoint mixture chrF++: 44.830004

Global checkpoint ranking:


,final_rank,candidate_name,candidate_type,continuation_step,macro_spBLEU,spBLEU_delta_vs_parent,macro_chrF++,chrF++_delta_vs_parent
0,1,devft_step_05200,periodic,5200,29.942952,2.905583,44.634356,1.733139
1,2,devft_step_05300,refinement,5300,29.928274,2.890905,44.592932,1.691715
2,3,devft_step_05100,refinement,5100,29.908029,2.870661,44.647495,1.746277
3,4,devft_step_04900,refinement,4900,29.885261,2.847892,44.590768,1.689551
4,5,devft_step_05400,refinement,5400,29.878675,2.841306,44.588679,1.687461
5,6,devft_step_05000,refinement,5000,29.860226,2.822857,44.621848,1.720630
6,7,devft_final_07845,final_adapter,7845,29.848937,2.811568,44.588605,1.687388
7,8,devft_step_07845,latest_periodic,7845,29.848937,2.811568,44.588605,1.687388
8,9,devft_step_06400,periodic,6400,29.839469,2.802100,44.571395,1.670178
9,10,devft_step_05600,periodic,5600,29.834589,2.797220,44.552618,1.651400



Best checkpoint independently for each country:


,country,best_candidate,best_continuation_step,selection_spBLEU,selection_chrF++
0,EG,devft_step_02000,2000,31.998721,45.780024
1,JO,devft_step_05000,5000,35.619749,49.370459
2,LB,devft_step_04800,4800,32.306780,45.955986
3,LY,devft_step_06400,6400,23.357296,39.104943
4,MA,devft_step_04900,4900,23.390521,39.750487
5,MR,devft_step_02000,2000,17.940801,34.338484
6,OM,devft_step_04900,4900,32.574507,47.108425
7,PS,devft_step_06000,6000,34.329883,48.332888
8,SA,devft_step_01600,1600,35.028283,49.698319
9,SD,devft_step_05200,5200,26.152188,40.973329



Saved global ranking: /home/mabdallah/alexandriax_mt_14d/checkpoint_selection/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1_beam4_training_style/results/final_checkpoint_ranking.csv
Saved per-country ranking: /home/mabdallah/alexandriax_mt_14d/checkpoint_selection/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1_beam4_training_style/results/best_checkpoint_per_country.csv
Saved best checkpoint: /home/mabdallah/alexandriax_mt_14d/checkpoint_selection/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1_beam4_training_style/results/best_checkpoint.json


### Gains against the best checkpoint

In [18]:
# ============================================================
# Cell 15A — Specialist checkpoint gains versus checkpoint 5200
# Produces the checkpoint shortlist for the variant stage
# ============================================================

DEFAULT_CANDIDATE = "devft_step_05200"
MIN_COUNTRY_SPBLEU_GAIN = 0.15
MAX_SPECIALIST_CHECKPOINTS = 2

required_columns = {
    "country",
    "candidate_name",
    "candidate_type",
    "continuation_step",
    "checkpoint_path",
    "spBLEU",
    "chrF++",
}

missing_columns = required_columns - set(all_checkpoint_country_metrics_df.columns)

if missing_columns:
    raise KeyError(
        f"all_checkpoint_country_metrics_df is missing: {sorted(missing_columns)}"
    )

metrics_df = all_checkpoint_country_metrics_df.copy()

metrics_df["country"] = metrics_df["country"].astype(str)
metrics_df["candidate_name"] = metrics_df["candidate_name"].astype(str)
metrics_df["continuation_step"] = pd.to_numeric(
    metrics_df["continuation_step"],
    errors="raise",
).astype(int)

metrics_df["spBLEU"] = pd.to_numeric(
    metrics_df["spBLEU"],
    errors="raise",
)

metrics_df["chrF++"] = pd.to_numeric(
    metrics_df["chrF++"],
    errors="raise",
)

countries = sorted(metrics_df["country"].unique())
number_of_countries = len(countries)

if number_of_countries != 13:
    raise RuntimeError(
        f"Expected 13 selection countries, found {number_of_countries}: {countries}"
    )

# ------------------------------------------------------------
# Checkpoint 5200 metrics for every country
# ------------------------------------------------------------

default_country_df = (
    metrics_df[
        metrics_df["candidate_name"] == DEFAULT_CANDIDATE
    ]
    .sort_values(
        ["country", "spBLEU", "chrF++"],
        ascending=[True, False, False],
    )
    .drop_duplicates(
        subset=["country"],
        keep="first",
    )
    .reset_index(drop=True)
)

if len(default_country_df) != number_of_countries:
    missing_default_countries = sorted(
        set(countries) - set(default_country_df["country"])
    )

    raise RuntimeError(
        f"{DEFAULT_CANDIDATE} is missing countries: "
        f"{missing_default_countries}"
    )

default_country_df = default_country_df[
    [
        "country",
        "candidate_name",
        "candidate_type",
        "continuation_step",
        "checkpoint_path",
        "spBLEU",
        "chrF++",
    ]
].rename(
    columns={
        "candidate_name": "default_candidate",
        "candidate_type": "default_candidate_type",
        "continuation_step": "default_step",
        "checkpoint_path": "default_checkpoint_path",
        "spBLEU": "default_spBLEU",
        "chrF++": "default_chrF++",
    }
)

# ------------------------------------------------------------
# Best checkpoint independently for each country under V01
# ------------------------------------------------------------

country_winner_df = (
    metrics_df
    .sort_values(
        [
            "country",
            "spBLEU",
            "chrF++",
            "continuation_step",
            "candidate_name",
        ],
        ascending=[
            True,
            False,
            False,
            True,
            True,
        ],
    )
    .drop_duplicates(
        subset=["country"],
        keep="first",
    )
    .reset_index(drop=True)
)

country_winner_df = country_winner_df[
    [
        "country",
        "candidate_name",
        "candidate_type",
        "continuation_step",
        "checkpoint_path",
        "spBLEU",
        "chrF++",
    ]
].rename(
    columns={
        "candidate_name": "best_candidate",
        "candidate_type": "best_candidate_type",
        "continuation_step": "best_step",
        "checkpoint_path": "best_checkpoint_path",
        "spBLEU": "best_spBLEU",
        "chrF++": "best_chrF++",
    }
)

checkpoint_gain_df = country_winner_df.merge(
    default_country_df,
    on="country",
    how="inner",
    validate="one_to_one",
)

checkpoint_gain_df["spBLEU_gain_vs_5200"] = (
    checkpoint_gain_df["best_spBLEU"]
    - checkpoint_gain_df["default_spBLEU"]
)

checkpoint_gain_df["chrF++_gain_vs_5200"] = (
    checkpoint_gain_df["best_chrF++"]
    - checkpoint_gain_df["default_chrF++"]
)

checkpoint_gain_df["is_different_checkpoint"] = (
    checkpoint_gain_df["best_candidate"]
    != DEFAULT_CANDIDATE
)

checkpoint_gain_df["passes_gain_threshold"] = (
    checkpoint_gain_df["is_different_checkpoint"]
    & (
        checkpoint_gain_df["spBLEU_gain_vs_5200"]
        >= MIN_COUNTRY_SPBLEU_GAIN
    )
)

checkpoint_gain_df["decision"] = np.select(
    [
        ~checkpoint_gain_df["is_different_checkpoint"],
        checkpoint_gain_df["passes_gain_threshold"],
    ],
    [
        "keep_5200",
        "eligible_specialist",
    ],
    default="gain_too_small",
)

checkpoint_gain_df = (
    checkpoint_gain_df
    .sort_values(
        [
            "spBLEU_gain_vs_5200",
            "chrF++_gain_vs_5200",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Summarize each specialist checkpoint
# ------------------------------------------------------------

specialist_records = []

nondefault_winners_df = checkpoint_gain_df[
    checkpoint_gain_df["is_different_checkpoint"]
].copy()

for candidate_name, candidate_rows in nondefault_winners_df.groupby(
    "best_candidate",
    sort=False,
):
    eligible_rows = candidate_rows[
        candidate_rows["passes_gain_threshold"]
    ].copy()

    specialist_records.append(
        {
            "candidate_name": candidate_name,
            "continuation_step": int(
                candidate_rows.iloc[0]["best_step"]
            ),
            "checkpoint_path": str(
                candidate_rows.iloc[0]["best_checkpoint_path"]
            ),
            "country_wins": int(len(candidate_rows)),
            "winning_countries": ",".join(
                sorted(candidate_rows["country"].astype(str))
            ),
            "eligible_country_wins": int(len(eligible_rows)),
            "eligible_countries": ",".join(
                sorted(eligible_rows["country"].astype(str))
            ),
            "total_spBLEU_gain": float(
                candidate_rows["spBLEU_gain_vs_5200"].sum()
            ),
            "eligible_total_spBLEU_gain": float(
                eligible_rows["spBLEU_gain_vs_5200"].sum()
            ),
            "eligible_macro_spBLEU_contribution": float(
                eligible_rows["spBLEU_gain_vs_5200"].sum()
                / number_of_countries
            ),
            "maximum_country_spBLEU_gain": float(
                candidate_rows["spBLEU_gain_vs_5200"].max()
            ),
            "mean_country_spBLEU_gain": float(
                candidate_rows["spBLEU_gain_vs_5200"].mean()
            ),
        }
    )

specialist_summary_df = pd.DataFrame(specialist_records)

if len(specialist_summary_df):
    specialist_summary_df = (
        specialist_summary_df
        .sort_values(
            [
                "eligible_macro_spBLEU_contribution",
                "eligible_country_wins",
                "maximum_country_spBLEU_gain",
                "continuation_step",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )
else:
    specialist_summary_df = pd.DataFrame(
        columns=[
            "candidate_name",
            "continuation_step",
            "checkpoint_path",
            "country_wins",
            "winning_countries",
            "eligible_country_wins",
            "eligible_countries",
            "total_spBLEU_gain",
            "eligible_total_spBLEU_gain",
            "eligible_macro_spBLEU_contribution",
            "maximum_country_spBLEU_gain",
            "mean_country_spBLEU_gain",
        ]
    )

eligible_specialist_df = (
    specialist_summary_df[
        specialist_summary_df["eligible_country_wins"] > 0
    ]
    .head(MAX_SPECIALIST_CHECKPOINTS)
    .copy()
)

selected_specialist_names = set(
    eligible_specialist_df["candidate_name"].astype(str)
)


# ------------------------------------------------------------
# Build the recommended V01-only pruned router
# This is diagnostic, not yet the final V03/V05 router
# ------------------------------------------------------------

pruned_router_records = []

for _, row in checkpoint_gain_df.iterrows():
    use_specialist = (
        row["best_candidate"] in selected_specialist_names
        and bool(row["passes_gain_threshold"])
    )

    if use_specialist:
        selected_candidate = str(row["best_candidate"])
        selected_step = int(row["best_step"])
        selected_checkpoint_path = str(row["best_checkpoint_path"])
        selected_spbleu = float(row["best_spBLEU"])
        selected_chrfpp = float(row["best_chrF++"])
        route_reason = "eligible_specialist"
    else:
        selected_candidate = str(row["default_candidate"])
        selected_step = int(row["default_step"])
        selected_checkpoint_path = str(row["default_checkpoint_path"])
        selected_spbleu = float(row["default_spBLEU"])
        selected_chrfpp = float(row["default_chrF++"])
        route_reason = "global_default_5200"

    pruned_router_records.append(
        {
            "country": str(row["country"]),
            "selected_candidate": selected_candidate,
            "selected_step": selected_step,
            "selected_checkpoint_path": selected_checkpoint_path,
            "selected_spBLEU": selected_spbleu,
            "selected_chrF++": selected_chrfpp,
            "spBLEU_gain_vs_5200": (
                selected_spbleu
                - float(row["default_spBLEU"])
            ),
            "route_reason": route_reason,
        }
    )

pruned_v01_router_df = (
    pd.DataFrame(pruned_router_records)
    .sort_values("country")
    .reset_index(drop=True)
)

default_macro_spbleu = float(
    default_country_df["default_spBLEU"].mean()
)

default_macro_chrfpp = float(
    default_country_df["default_chrF++"].mean()
)

full_checkpoint_mixture_spbleu = float(
    checkpoint_gain_df["best_spBLEU"].mean()
)

full_checkpoint_mixture_chrfpp = float(
    checkpoint_gain_df["best_chrF++"].mean()
)

pruned_router_macro_spbleu = float(
    pruned_v01_router_df["selected_spBLEU"].mean()
)

pruned_router_macro_chrfpp = float(
    pruned_v01_router_df["selected_chrF++"].mean()
)

full_mixture_gain = (
    full_checkpoint_mixture_spbleu
    - default_macro_spbleu
)

pruned_mixture_gain = (
    pruned_router_macro_spbleu
    - default_macro_spbleu
)

recovered_uplift_fraction = (
    pruned_mixture_gain / full_mixture_gain
    if full_mixture_gain > 0
    else 0.0
)

# ------------------------------------------------------------
# Final checkpoint shortlist for V03/V05 generation
# ------------------------------------------------------------

default_checkpoint_row = default_country_df.iloc[0]

shortlist_records = [
    {
        "candidate_name": DEFAULT_CANDIDATE,
        "continuation_step": int(
            default_checkpoint_row["default_step"]
        ),
        "checkpoint_path": str(
            default_checkpoint_row["default_checkpoint_path"]
        ),
        "role": "global_default",
        "variant_stage_countries": ",".join(countries),
        "variants_to_test": "V01,V03,V05",
    }
]

for specialist in eligible_specialist_df.itertuples(index=False):
    shortlist_records.append(
        {
            "candidate_name": specialist.candidate_name,
            "continuation_step": int(
                specialist.continuation_step
            ),
            "checkpoint_path": str(
                specialist.checkpoint_path
            ),
            "role": "country_specialist",
            "variant_stage_countries": specialist.eligible_countries,
            "variants_to_test": "V01,V03,V05",
        }
    )

variant_checkpoint_shortlist_df = pd.DataFrame(
    shortlist_records
)

# ------------------------------------------------------------
# Save all diagnostic outputs
# ------------------------------------------------------------

GAIN_DETAILS_PATH = (
    SELECTION_RESULTS_DIR
    / "checkpoint_country_gains_vs_step5200.csv"
)

SPECIALIST_SUMMARY_PATH = (
    SELECTION_RESULTS_DIR
    / "checkpoint_specialist_summary_vs_step5200.csv"
)

VARIANT_SHORTLIST_PATH = (
    SELECTION_RESULTS_DIR
    / "checkpoint_shortlist_for_variant_stage.csv"
)

PRUNED_ROUTER_PATH = (
    SELECTION_RESULTS_DIR
    / "pruned_checkpoint_router_v01.csv"
)

SPECIALIST_DIAGNOSTIC_PATH = (
    SELECTION_RESULTS_DIR
    / "checkpoint_specialist_diagnostic.json"
)

checkpoint_gain_df.to_csv(
    GAIN_DETAILS_PATH,
    index=False,
    encoding="utf-8-sig",
)

specialist_summary_df.to_csv(
    SPECIALIST_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

variant_checkpoint_shortlist_df.to_csv(
    VARIANT_SHORTLIST_PATH,
    index=False,
    encoding="utf-8-sig",
)

pruned_v01_router_df.to_csv(
    PRUNED_ROUTER_PATH,
    index=False,
    encoding="utf-8-sig",
)

diagnostic_payload = {
    "default_candidate": DEFAULT_CANDIDATE,
    "minimum_country_spBLEU_gain": MIN_COUNTRY_SPBLEU_GAIN,
    "maximum_specialist_checkpoints": MAX_SPECIALIST_CHECKPOINTS,
    "selected_specialists": sorted(selected_specialist_names),
    "default_macro_spBLEU": default_macro_spbleu,
    "default_macro_chrF++": default_macro_chrfpp,
    "full_checkpoint_mixture_macro_spBLEU": full_checkpoint_mixture_spbleu,
    "full_checkpoint_mixture_macro_chrF++": full_checkpoint_mixture_chrfpp,
    "full_checkpoint_mixture_gain": full_mixture_gain,
    "pruned_router_macro_spBLEU": pruned_router_macro_spbleu,
    "pruned_router_macro_chrF++": pruned_router_macro_chrfpp,
    "pruned_router_gain": pruned_mixture_gain,
    "recovered_uplift_fraction": recovered_uplift_fraction,
}

with open(
    SPECIALIST_DIAGNOSTIC_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        diagnostic_payload,
        file,
        ensure_ascii=False,
        indent=2,
    )

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 90)
print("COUNTRY CHECKPOINT GAINS VERSUS CHECKPOINT 5200")
print("=" * 90)

display(
    checkpoint_gain_df[
        [
            "country",
            "best_candidate",
            "best_step",
            "best_spBLEU",
            "default_spBLEU",
            "spBLEU_gain_vs_5200",
            "best_chrF++",
            "default_chrF++",
            "chrF++_gain_vs_5200",
            "decision",
        ]
    ]
)

print("\nSpecialist checkpoint summary:")
display(specialist_summary_df)

print("\nRecommended checkpoints for the variant stage:")
display(variant_checkpoint_shortlist_df)

print("\nDiagnostic pruned V01 router:")
display(
    pruned_v01_router_df[
        [
            "country",
            "selected_candidate",
            "selected_step",
            "selected_spBLEU",
            "selected_chrF++",
            "spBLEU_gain_vs_5200",
            "route_reason",
        ]
    ]
)

print("\nMacro comparison:")
print(f"Checkpoint 5200 only:        {default_macro_spbleu:.6f}")
print(f"All checkpoint specialists: {full_checkpoint_mixture_spbleu:.6f}")
print(f"Pruned specialist router:   {pruned_router_macro_spbleu:.6f}")
print(f"Full mixture gain:          {full_mixture_gain:+.6f}")
print(f"Pruned router gain:         {pruned_mixture_gain:+.6f}")
print(f"Recovered mixture uplift:   {recovered_uplift_fraction:.2%}")

print("\nSaved:")
print(GAIN_DETAILS_PATH)
print(SPECIALIST_SUMMARY_PATH)
print(VARIANT_SHORTLIST_PATH)
print(PRUNED_ROUTER_PATH)
print(SPECIALIST_DIAGNOSTIC_PATH)

print(
    "\nThis is a V01 checkpoint diagnostic only. "
    "Do not freeze the final country router until V03 and V05 are evaluated."
)

COUNTRY CHECKPOINT GAINS VERSUS CHECKPOINT 5200


,country,best_candidate,best_step,best_spBLEU,default_spBLEU,spBLEU_gain_vs_5200,best_chrF++,default_chrF++,chrF++_gain_vs_5200,decision
0,OM,devft_step_04900,4900,32.574507,31.841459,0.733048,47.108425,46.527519,0.580906,eligible_specialist
1,LY,devft_step_06400,6400,23.357296,22.856993,0.500304,39.104943,38.586206,0.518737,eligible_specialist
2,YE,devft_step_04900,4900,25.233358,24.746359,0.486999,41.705242,41.159683,0.545559,eligible_specialist
3,SA,devft_step_01600,1600,35.028283,34.730874,0.297409,49.698319,49.580115,0.118205,eligible_specialist
4,EG,devft_step_02000,2000,31.998721,31.731002,0.267719,45.780024,45.487254,0.292770,eligible_specialist
5,MA,devft_step_04900,4900,23.390521,23.126635,0.263886,39.750487,39.613185,0.137302,eligible_specialist
6,MR,devft_step_02000,2000,17.940801,17.760150,0.180651,34.338484,34.387167,-0.048682,eligible_specialist
7,TN,devft_step_07600,7600,35.229468,35.064940,0.164528,47.608618,47.452205,0.156413,eligible_specialist
8,JO,devft_step_05000,5000,35.619749,35.499177,0.120572,49.370459,49.122497,0.247962,gain_too_small
9,PS,devft_step_06000,6000,34.329883,34.256767,0.073116,48.332888,48.258044,0.074844,gain_too_small



Specialist checkpoint summary:


,candidate_name,continuation_step,checkpoint_path,country_wins,winning_countries,eligible_country_wins,eligible_countries,total_spBLEU_gain,eligible_total_spBLEU_gain,eligible_macro_spBLEU_contribution,maximum_country_spBLEU_gain,mean_country_spBLEU_gain
0,devft_step_04900,4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,3,"MA,OM,YE",3,"MA,OM,YE",1.483934,1.483934,0.114149,0.733048,0.494645
1,devft_step_06400,6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,2,"LY,SY",1,LY,0.539484,0.500304,0.038485,0.500304,0.269742
2,devft_step_02000,2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,2,"EG,MR",2,"EG,MR",0.448370,0.448370,0.034490,0.267719,0.224185
3,devft_step_01600,1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,1,SA,1,SA,0.297409,0.297409,0.022878,0.297409,0.297409
4,devft_step_07600,7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,1,TN,1,TN,0.164528,0.164528,0.012656,0.164528,0.164528
5,devft_step_05000,5000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,1,JO,0,,0.120572,0.000000,0.000000,0.120572,0.120572
6,devft_step_06000,6000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,1,PS,0,,0.073116,0.000000,0.000000,0.073116,0.073116
7,devft_step_04800,4800,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,1,LB,0,,0.018680,0.000000,0.000000,0.018680,0.018680



Recommended checkpoints for the variant stage:


,candidate_name,continuation_step,checkpoint_path,role,variant_stage_countries,variants_to_test
0,devft_step_05200,5200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,global_default,"EG,JO,LB,LY,MA,MR,OM,PS,SA,SD,SY,TN,YE","V01,V03,V05"
1,devft_step_04900,4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,country_specialist,"MA,OM,YE","V01,V03,V05"
2,devft_step_06400,6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...,country_specialist,LY,"V01,V03,V05"



Diagnostic pruned V01 router:


,country,selected_candidate,selected_step,selected_spBLEU,selected_chrF++,spBLEU_gain_vs_5200,route_reason
0,EG,devft_step_05200,5200,31.731002,45.487254,0.000000,global_default_5200
1,JO,devft_step_05200,5200,35.499177,49.122497,0.000000,global_default_5200
2,LB,devft_step_05200,5200,32.288100,46.000736,0.000000,global_default_5200
3,LY,devft_step_06400,6400,23.357296,39.104943,0.500304,eligible_specialist
4,MA,devft_step_04900,4900,23.390521,39.750487,0.263886,eligible_specialist
5,MR,devft_step_05200,5200,17.760150,34.387167,0.000000,global_default_5200
6,OM,devft_step_04900,4900,32.574507,47.108425,0.733048,eligible_specialist
7,PS,devft_step_05200,5200,34.256767,48.258044,0.000000,global_default_5200
8,SA,devft_step_05200,5200,34.730874,49.580115,0.000000,global_default_5200
9,SD,devft_step_05200,5200,26.152188,40.973329,0.000000,global_default_5200



Macro comparison:
Checkpoint 5200 only:        29.942952
All checkpoint specialists: 30.184959
Pruned specialist router:   30.095586
Full mixture gain:          +0.242007
Pruned router gain:         +0.152634
Recovered mixture uplift:   63.07%

Saved:
/home/mabdallah/alexandriax_mt_14d/checkpoint_selection/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1_beam4_training_style/results/checkpoint_country_gains_vs_step5200.csv
/home/mabdallah/alexandriax_mt_14d/checkpoint_selection/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1_beam4_training_style/results/checkpoint_specialist_summary_vs_step5200.csv
/home/mabdallah/alexandriax_mt_14d/checkpoint_selection/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1_beam4_training_style/results/checkpoint_shortlist_for_variant_stage.csv
/home/mabdallah/alexandriax_mt_14d/checkpoint_selection/nilechat3b_dev1225